# Colombia Saludable: diagnóstico previo al ETL con el Core

Este cuaderno comprueba las conexiones, documenta el origen y reúne evidencia para el diseño de la bodega.
**La parte A (secciones 1–9) es de solo lectura. La parte B implementa el ETL y solo escribe si se activa explícitamente la carga.**
Las consultas se ejecutan mediante `etl.DB_Extractor`; SQLAlchemy solo configura las sesiones de lectura.
Los DataFrames y CSV contienen metadatos y agregados de diagnóstico, sin muestras de personas ni credenciales.

Ejecutar de arriba abajo con el Python de `.venv`. El archivo `colombia_saludable/.env` es la única fuente de conexión.
Las salidas guardadas son una fotografía del momento de ejecución; cada consulta abre su propia transacción,
por lo que el conjunto completo no representa un único snapshot si el origen cambia durante el análisis.

**Funciones utilizadas en la siguiente celda**

- **Core, directamente o a través de auxiliares:** `CSV_Loader.save_csv`, `DB_Extractor.execute_query`, `etl.CSV_Loader`.
- **Fuera del Core y motivo:** Path, sys, dotenv, datetime, ZoneInfo y metadatos de paquetes configuran el entorno; pandas crea tablas de evidencia e IPython las muestra. JSON y archivos conservan manifiestos propios.

**Funciones propias definidas o llamadas en esta celda**

| Función | Relación con el Core y motivo de conservarla |
| --- | --- |
| `guardar` | delega el CSV al Core y conserva el registro de evidencias y la presentación IPython. |
| `ident` | escapa identificadores SQL del catálogo; no calcula medidas. |
| `q` | delega el SQL a DB_Extractor.execute_query y silencia sus mensajes. |
| `tabla` | construye el nombre SQL public.tabla con identificadores escapados. |


**Llamadas externas directas de esta celda** (incluye los cuerpos de funciones definidos aquí; las dependencias de auxiliares llamados se explican donde se definen; `…` omite argumentos):

| Grupo | Funciones y métodos | Por qué se conservan fuera del Core |
| --- | --- | --- |
| Python estándar y contenedores | `RuntimeError`, `ValueError`, `cfg.get`, `next`, `print`, `str`, `str.join`, `valor.replace` | Orquestación, condiciones, iteración, construcción de SQL, parámetros y mensajes. No sustituyen una transformación tabular del Core. |
| Archivos, configuración y reproducibilidad | `EVIDENCIA.mkdir`, `Path.cwd`, `ZoneInfo`, `contextlib.redirect_stdout`, `datetime.now`, `datetime.now(…).isoformat`, `dotenv_values`, `io.StringIO`, `platform.python_version`, `ruta.is_file`, `sys.path.insert` | Rutas, entorno, manifiestos, fechas de ejecución, versiones o hashes específicos de este proceso; el Core no publica estos manifiestos del taller. |
| Dibujo e IPython | `display` | Renderizado, ejes, leyendas, imágenes y visualización del notebook; el Core no ofrece renderizadores Python de gráficos ni display HTML. |

**Estado de esta versión:** ejecutada el 30 de octubre de 2026 sobre `colombia_saludable_dw_core_v2`. Las salidas guardadas corresponden a esta ejecución; las evidencias anteriores se conservaron en `../historial_ejecuciones/20261030T150841/`.


In [1]:
# Preparación del diagnóstico y utilidades compartidas.

from pathlib import Path
import sys
import json
import contextlib
import io
import platform
from datetime import datetime
from zoneinfo import ZoneInfo
from importlib.metadata import version
import pandas as pd
from IPython.display import display, Markdown
from dotenv import dotenv_values
from sqlalchemy import event

# Funciona desde la raíz del repositorio o desde cuadernos/.

# Localizar el repositorio para permitir ejecución desde la raíz o desde cuadernos/.
ROOT = next(
    (
        p
        for p in [Path.cwd(), *Path.cwd().parents]
        if (p / "etl/extractors/db_extractor.py").is_file()
        and (p / "colombia_saludable/.env").is_file()
    ),
    None,
)
if ROOT is None:
    raise RuntimeError("Abra el cuaderno dentro de este repositorio.")
sys.path.insert(0, str(ROOT)) if str(ROOT) not in sys.path else None
from etl import (
    DB_Extractor,
    CSV_Loader,
    BasicsTransformOperations as B,
    DataSelect as S,
    TransformOperations as T,
)
from etl.schema import missing as nulo

# Preparar el loader CSV del Core y las carpetas de evidencia.
CSV_CORE = CSV_Loader()

BASE = ROOT / "colombia_saludable"
EVIDENCIA = BASE / "planes_y_evidencia"
EVIDENCIA.mkdir(exist_ok=True)

# Leer las variables de conexión y comprobar que origen y destino sean distintos.
cfg = dotenv_values(BASE / ".env")
requeridas = [
    "DB_TYPE",
    "DB_HOST",
    "DB_PORT",
    "DB_USER",
    "DB_PASSWORD",
    "DB_NAME_SOURCE",
    "DB_NAME_WAREHOUSE",
]
faltantes = [k for k in requeridas if not cfg.get(k)]
if faltantes:
    raise ValueError("Variables faltantes: " + ", ".join(faltantes))
if cfg["DB_TYPE"] != "postgresql" or cfg["DB_NAME_SOURCE"] == cfg["DB_NAME_WAREHOUSE"]:
    raise ValueError("Se requiere PostgreSQL y bases de origen y destino distintas.")

# Registrar la fecha de inicio y reunir los agregados que se exportarán.
INICIO = datetime.now(ZoneInfo("America/Bogota")).isoformat()
evidencias = {}


def guardar(nombre, frame, mostrar=True):
    # Registrar el agregado y exportarlo con el Core antes de mostrarlo.
    evidencias[nombre] = frame
    CSV_CORE.save_csv(
        frame,
        str(EVIDENCIA / f"{nombre}.csv"),
        index=False,
        encoding="utf-8-sig",
        lineterminator='\n',
    )
    if mostrar:
        display(frame)


def q(extractor, sql):
    # Evita mensajes repetidos del Core; nunca imprime la configuración.
    # Conservar la ejecución SQL del Core sin repetir sus mensajes en cada consulta.
    with contextlib.redirect_stdout(io.StringIO()):
        return extractor.execute_query(sql)


def ident(valor):
    # Escapar comillas del identificador antes de incorporarlo al SQL.
    return '"' + valor.replace('"', '""') + '"'


def tabla(nombre):
    # Construir un nombre SQL del esquema público con el identificador escapado.
    return '"public".' + ident(nombre)


print("Inicio del diagnóstico:", INICIO)
print("Python:", platform.python_version())


Inicio del diagnóstico: 2026-10-30T15:08:50.019084-05:00
Python: 3.14.0


## 1. Comprobación de origen y bodega

Se valida la conectividad con `connect()` y `check_connection_status()` del Core y se consulta la identidad real
de cada base. Tras la conexión inicial se renueva el pool para que todas las lecturas usen sesiones
`READ ONLY`, con límites de 30 segundos por sentencia y 5 segundos para bloqueos.
La conexión exitosa no certifica permisos de escritura: no se prueban porque todavía no se autoriza cargar datos.

**Funciones utilizadas en la siguiente celda**

- **Core, directamente o a través de auxiliares:** `CSV_Loader.save_csv`, `DB_Extractor.check_connection_status`, `DB_Extractor.close_connection`, `DB_Extractor.connect`, `DB_Extractor.execute_query`, `etl.DB_Extractor`.
- **Fuera del Core y motivo:** SQLAlchemy event.listens_for, set_session, exec_driver_sql y engine.dispose configuran sesiones. pandas construye el resumen de conexiones; contextlib oculta mensajes.

**Funciones propias definidas o llamadas en esta celda**

| Función | Relación con el Core y motivo de conservarla |
| --- | --- |
| `configurar_lectura` | configura READ ONLY y aislamiento mediante el driver, fuera de la API de DB_Extractor. |
| `guardar` | delega el CSV al Core y conserva el registro de evidencias y la presentación IPython. |
| `limitar_consulta` | configura timeouts mediante eventos de SQLAlchemy. |
| `q` | delega el SQL a DB_Extractor.execute_query y silencia sus mensajes. |

**SQL:** las consultas de esta celda se ejecutan mediante `DB_Extractor.execute_query`, pero sus expresiones SQL pertenecen al cuaderno. Se conserva SQL para perfilado, validación o conciliación; la extracción B1 solo selecciona columnas y la transformación tabular de B2–B5 usa los transformadores Core.

**Llamadas externas directas de esta celda** (incluye los cuerpos de funciones definidos aquí; las dependencias de auxiliares llamados se explican donde se definen; `…` omite argumentos):

| Grupo | Funciones y métodos | Por qué se conservan fuera del Core |
| --- | --- | --- |
| Python estándar y contenedores | `RuntimeError`, `conexiones.append`, `extractores.values`, `int`, `type` | Orquestación, condiciones, iteración, construcción de SQL, parámetros y mensajes. No sustituyen una transformación tabular del Core. |
| Archivos, configuración y reproducibilidad | `contextlib.redirect_stdout`, `io.StringIO` | Rutas, entorno, manifiestos, fechas de ejecución, versiones o hashes específicos de este proceso; el Core no publica estos manifiestos del taller. |
| Sesiones SQLAlchemy y driver | `connection.exec_driver_sql`, `dbapi_connection.set_session`, `event.listens_for`, `ex.engine.dispose` | Modo de lectura, aislamiento y tiempos límite añadidos a DB_Extractor; no son cargas de datos por fuera del Core. |
| DataFrames y matrices | `pd.DataFrame`, `q(…).iloc[…].to_dict` | Construcción de evidencia, acceso/iteración de resultados, selección de máximos y presentación de matrices. Los filtros, CSV, conversiones y deduplicación con equivalencia directa se delegan al Core. head del Core devuelve texto; no hay pivot/reindex tabular equivalente en estos transformadores. |

In [2]:
# Comprobación de conexiones al origen y a la bodega.


# Conservar ambos extractores para reutilizarlos y cerrar todos sus pools.
extractores = {}
conexiones = []
try:
    for rol, clave in [("origen", "DB_NAME_SOURCE"), ("bodega", "DB_NAME_WAREHOUSE")]:
        ex = DB_Extractor(
            db_type=cfg["DB_TYPE"],
            host=cfg["DB_HOST"],
            port=int(cfg["DB_PORT"]),
            user=cfg["DB_USER"],
            password=cfg["DB_PASSWORD"],
            database=cfg[clave],
            connect_timeout=5,
        )
        extractores[rol] = ex
        with contextlib.redirect_stdout(io.StringIO()):
            ex.connect()

        @event.listens_for(ex.engine, "connect")
        def configurar_lectura(dbapi_connection, connection_record):
            # Proteger cada conexión nueva antes de ejecutar las consultas de
            # diagnóstico.
            dbapi_connection.set_session(
                readonly=True, isolation_level="REPEATABLE READ", autocommit=False
            )

        @event.listens_for(ex.engine, "begin")
        def limitar_consulta(connection):
            # Acotar tanto el tiempo de ejecución como la espera por bloqueos.
            connection.exec_driver_sql("SET LOCAL statement_timeout = '30s'")
            connection.exec_driver_sql("SET LOCAL lock_timeout = '5s'")

        ex.engine.dispose()
        if not ex.check_connection_status():
            raise RuntimeError("Fallo de comprobación de conexión")
        registro = (
            q(
                ex,
                """SELECT current_database() AS base, current_user AS usuario,
            current_schema() AS esquema, current_setting('server_version') AS version_postgresql,
            current_setting('transaction_read_only') AS solo_lectura,
            current_setting('TimeZone') AS zona_horaria""",
            )
            .iloc[0]
            .to_dict()
        )
        assert registro["base"] == cfg[clave] and registro["solo_lectura"] == "on"
        assert registro["esquema"] == "public", "Revisar esquema antes del perfilado"
        conexiones.append({"rol": rol, "estado": "OK", **registro})
except Exception as exc:
    for ex in extractores.values():
        with contextlib.redirect_stdout(io.StringIO()):
            ex.close_connection()
    raise RuntimeError(
        "No fue posible validar ambas conexiones. Revise .env y el servidor local. "
        + type(exc).__name__
    ) from None
guardar("conexiones", pd.DataFrame(conexiones))
origen, bodega = extractores["origen"], extractores["bodega"]


💾 CSV guardado exitosamente: C:\Users\cabre\Desktop\University\Tesis\python-library-etl-master\python-library-etl-master\colombia_saludable\planes_y_evidencia\conexiones.csv (2 registros, 8 columnas)


,rol,estado,base,usuario,esquema,version_postgresql,solo_lectura,zona_horaria
0,origen,OK,colombia_saludable,postgres,public,18.3,on,America/Bogota
1,bodega,OK,colombia_saludable_dw_core_v2,postgres,public,18.3,on,America/Bogota


## 2. Catálogo, diccionario de datos y volumen

Se inventarían todos los esquemas no internos para no confundir un `public` vacío con una base vacía.
El perfilado detallado usa las tablas físicas de `public`, verificadas mediante el catálogo del Core.
Los conteos son exactos; las PK y FK se recuperan con `get_table_schema()`.

**Funciones utilizadas en la siguiente celda**

- **Core, directamente o a través de auxiliares:** `CSV_Loader.save_csv`, `DB_Extractor.execute_query`, `DB_Extractor.get_table_names`, `DB_Extractor.get_table_schema`.
- **Fuera del Core y motivo:** SQL de information_schema y COUNT define el inventario; Python organiza metadatos y pandas construye las tablas de evidencia.

**Funciones propias definidas o llamadas en esta celda**

| Función | Relación con el Core y motivo de conservarla |
| --- | --- |
| `guardar` | delega el CSV al Core y conserva el registro de evidencias y la presentación IPython. |
| `q` | delega el SQL a DB_Extractor.execute_query y silencia sus mensajes. |
| `tabla` | construye el nombre SQL public.tabla con identificadores escapados. |

**SQL:** las consultas de esta celda se ejecutan mediante `DB_Extractor.execute_query`, pero sus expresiones SQL pertenecen al cuaderno. Se conserva SQL para perfilado, validación o conciliación; la extracción B1 solo selecciona columnas y la transformación tabular de B2–B5 usa los transformadores Core.

**Llamadas externas directas de esta celda** (incluye los cuerpos de funciones definidos aquí; las dependencias de auxiliares llamados se explican donde se definen; `…` omite argumentos):

| Grupo | Funciones y métodos | Por qué se conservan fuera del Core |
| --- | --- | --- |
| Python estándar y contenedores | `extractores.items`, `int`, `len`, `sorted`, `str.join`, `volumen.append` | Orquestación, condiciones, iteración, construcción de SQL, parámetros y mensajes. No sustituyen una transformación tabular del Core. |
| Archivos, configuración y reproducibilidad | `contextlib.redirect_stdout`, `io.StringIO` | Rutas, entorno, manifiestos, fechas de ejecución, versiones o hashes específicos de este proceso; el Core no publica estos manifiestos del taller. |
| DataFrames y matrices | `pd.DataFrame` | Construcción de evidencia, acceso/iteración de resultados, selección de máximos y presentación de matrices. Los filtros, CSV, conversiones y deduplicación con equivalencia directa se delegan al Core. head del Core devuelve texto; no hay pivot/reindex tabular equivalente en estos transformadores. |
| Dibujo e IPython | `Markdown`, `display` | Renderizado, ejes, leyendas, imágenes y visualización del notebook; el Core no ofrece renderizadores Python de gráficos ni display HTML. |

In [3]:
# Inventario del catálogo, las columnas y el volumen del origen.


# Incluir esquemas de usuario para no confundir un public vacío con una base vacía.
catalogo_sql = """SELECT table_schema AS esquema, table_name AS tabla, table_type AS tipo
FROM information_schema.tables
WHERE table_schema NOT IN ('information_schema', 'pg_catalog')
  AND table_schema NOT LIKE 'pg_toast%' AND table_schema NOT LIKE 'pg_temp%'
ORDER BY table_schema, table_name"""

# Guardar la fotografía estructural inicial de las dos bases.
inventarios_iniciales = {rol: q(ex, catalogo_sql) for rol, ex in extractores.items()}
guardar("catalogo_origen", inventarios_iniciales["origen"])
guardar("catalogo_bodega_inicial", inventarios_iniciales["bodega"])

# Consultar tipos, nulabilidad, tamaños y valores predeterminados de las columnas.
columnas = q(
    origen,
    """SELECT table_name AS tabla, ordinal_position AS posicion, column_name AS columna,
    data_type AS tipo, udt_name AS tipo_interno, is_nullable AS admite_nulo,
    character_maximum_length AS longitud_maxima, numeric_precision AS precision,
    numeric_scale AS escala, column_default AS valor_predeterminado
    FROM information_schema.columns WHERE table_schema='public' ORDER BY table_name, ordinal_position""",
)
guardar("diccionario_origen", columnas, mostrar=False)
with contextlib.redirect_stdout(io.StringIO()):
    nombres = sorted(origen.get_table_names())
    esquemas = {t: origen.get_table_schema(t) for t in nombres}

# Combinar los conteos exactos con las claves recuperadas por el Core.
volumen = []
for t in nombres:
    filas = int(q(origen, f"SELECT COUNT(*) AS n FROM {tabla(t)}").iloc[0, 0])
    volumen.append(
        {
            "tabla": t,
            "filas": filas,
            "columnas": len(esquemas[t]["columns"]),
            "pk": ", ".join(esquemas[t]["primary_keys"]),
            "fk_declaradas": len(esquemas[t]["foreign_keys"]),
        }
    )
guardar("volumen_origen", pd.DataFrame(volumen))
display(
    Markdown("Diccionario completo: `../planes_y_evidencia/diccionario_origen.csv`.")
)


💾 CSV guardado exitosamente: C:\Users\cabre\Desktop\University\Tesis\python-library-etl-master\python-library-etl-master\colombia_saludable\planes_y_evidencia\catalogo_origen.csv (18 registros, 3 columnas)


,esquema,tabla,tipo
0,public,beneficiario,BASE TABLE
1,public,citas_generales,BASE TABLE
2,public,cotizante,BASE TABLE
3,public,cotizante_beneficiario,BASE TABLE
4,public,drogueria,BASE TABLE
5,public,empresa,BASE TABLE
6,public,empresa_cotizante,BASE TABLE
7,public,formulas_medicas,BASE TABLE
8,public,hospitalizaciones,BASE TABLE
9,public,ips,BASE TABLE


💾 CSV guardado exitosamente: C:\Users\cabre\Desktop\University\Tesis\python-library-etl-master\python-library-etl-master\colombia_saludable\planes_y_evidencia\catalogo_bodega_inicial.csv (0 registros, 3 columnas)


,esquema,tabla,tipo


💾 CSV guardado exitosamente: C:\Users\cabre\Desktop\University\Tesis\python-library-etl-master\python-library-etl-master\colombia_saludable\planes_y_evidencia\diccionario_origen.csv (107 registros, 10 columnas)


💾 CSV guardado exitosamente: C:\Users\cabre\Desktop\University\Tesis\python-library-etl-master\python-library-etl-master\colombia_saludable\planes_y_evidencia\volumen_origen.csv (18 registros, 5 columnas)


,tabla,filas,columnas,pk,fk_declaradas
0,beneficiario,3316,8,id_beneficiario,0
1,citas_generales,3013,8,codigo_cita,1
2,cotizante,1100,14,cedula,1
3,cotizante_beneficiario,3316,2,,2
4,drogueria,35,5,id_drogueria,0
5,empresa,21,6,nit,0
6,empresa_cotizante,925,2,"empresa, cotizante",2
7,formulas_medicas,7484,5,codigo_formula,1
8,hospitalizaciones,893,9,codigo_hospitalizacion,1
9,ips,86,7,id_ips,0


Diccionario completo: `../planes_y_evidencia/diccionario_origen.csv`.

## 3. Calidad por columna y duplicados

Se cuentan nulos SQL, textos vacíos después de `TRIM`, cardinalidad no nula y rangos de fechas/números.
`filas_duplicadas_excedentes` cuenta las copias adicionales de registros idénticos; no se eliminan aquí.
Las tablas sin PK requieren una regla de grano antes de cargar.

**Funciones utilizadas en la siguiente celda**

- **Core, directamente o a través de auxiliares:** `BasicsTransformOperations.filter_by_condition`, `CSV_Loader.save_csv`, `DB_Extractor.execute_query`, `DataSelect.filter_equal`.
- **Fuera del Core y motivo:** Las expresiones SQL COUNT/FILTER/MIN/MAX y GROUP BY definen el perfilado exacto en origen. Python arma esas expresiones y los registros de evidencia. El filtro OR del resultado se delega al Core.

**Funciones propias definidas o llamadas en esta celda**

| Función | Relación con el Core y motivo de conservarla |
| --- | --- |
| `guardar` | delega el CSV al Core y conserva el registro de evidencias y la presentación IPython. |
| `ident` | escapa identificadores SQL del catálogo; no calcula medidas. |
| `q` | delega el SQL a DB_Extractor.execute_query y silencia sus mensajes. |
| `tabla` | construye el nombre SQL public.tabla con identificadores escapados. |

**Expresiones de fila:** 1 lambda(s) escritas aquí. Su fórmula o condición pertenece al cuaderno; cuando se pasan a un transformador, el Core realiza la operación tabular. No se presentan como funciones propias del Core.

**SQL:** las consultas de esta celda se ejecutan mediante `DB_Extractor.execute_query`, pero sus expresiones SQL pertenecen al cuaderno. Se conserva SQL para perfilado, validación o conciliación; la extracción B1 solo selecciona columnas y la transformación tabular de B2–B5 usa los transformadores Core.

**Llamadas externas directas de esta celda** (incluye los cuerpos de funciones definidos aquí; las dependencias de auxiliares llamados se explican donde se definen; `…` omite argumentos):

| Grupo | Funciones y métodos | Por qué se conservan fuera del Core |
| --- | --- | --- |
| Python estándar y contenedores | `bool`, `datos.get`, `duplicados.append`, `enumerate`, `int`, `perfil.append`, `str`, `str.join` | Orquestación, condiciones, iteración, construcción de SQL, parámetros y mensajes. No sustituyen una transformación tabular del Core. |
| DataFrames y matrices | `S.filter_equal(…).to_dict`, `pd.DataFrame` | Construcción de evidencia, acceso/iteración de resultados, selección de máximos y presentación de matrices. Los filtros, CSV, conversiones y deduplicación con equivalencia directa se delegan al Core. head del Core devuelve texto; no hay pivot/reindex tabular equivalente en estos transformadores. |
| Dibujo e IPython | `display` | Renderizado, ejes, leyendas, imágenes y visualización del notebook; el Core no ofrece renderizadores Python de gráficos ni display HTML. |

In [4]:
# Perfilado de calidad sin modificar ni descartar filas del origen.


# Acumular métricas por columna y duplicados excedentes por tabla.
perfil = []
duplicados = []
for t in nombres:

    # Consultar todas las métricas de una tabla en una sola sentencia agregada.
    cols = S.filter_equal(columnas, "tabla", t, show=0).to_dict("records")
    expresiones = []
    for i, c in enumerate(cols):
        col = ident(c["columna"])
        expresiones += [
            f'COUNT(*) FILTER (WHERE {col} IS NULL) AS n{i}',
            f'COUNT(DISTINCT {col}) AS d{i}',
        ]
        if c["tipo"] in ("character varying", "text", "character"):
            expresiones += [
                f"COUNT(*) FILTER (WHERE {col} IS NOT NULL AND BTRIM({col}) = '') AS"
                f" v{i}"
            ]
        if c["tipo"] in (
            "date",
            "time without time zone",
            "timestamp without time zone",
            "integer",
            "bigint",
            "numeric",
        ):
            expresiones += [f'MIN({col}) AS min{i}', f'MAX({col}) AS max{i}']
    datos = q(origen, f'SELECT {", ".join(expresiones)} FROM {tabla(t)}').iloc[0]
    for i, c in enumerate(cols):
        perfil.append(
            {
                "tabla": t,
                "columna": c["columna"],
                "tipo": c["tipo"],
                "nulos": int(datos[f"n{i}"]),
                "distintos_no_nulos": int(datos[f"d{i}"]),
                "vacios_trim": int(datos.get(f"v{i}", 0)),
                "minimo": str(datos.get(f"min{i}", "")),
                "maximo": str(datos.get(f"max{i}", "")),
            }
        )

    # Agrupar por todas las columnas para contar copias adicionales de filas idénticas.
    lista = ', '.join(ident(c) for c in esquemas[t]["columns"])
    d = q(
        origen,
        'SELECT COALESCE(SUM(n-1),0)::bigint AS excedentes FROM '
        f'(SELECT COUNT(*) AS n FROM {tabla(t)} GROUP BY {lista} HAVING COUNT(*)>1) x',
    ).iloc[0, 0]
    duplicados.append(
        {
            "tabla": t,
            "tiene_pk": bool(esquemas[t]["primary_keys"]),
            "filas_duplicadas_excedentes": int(d),
        }
    )
guardar("perfil_columnas", pd.DataFrame(perfil), mostrar=False)
guardar("duplicados", pd.DataFrame(duplicados))
display(
    B.filter_by_condition(
        evidencias["perfil_columnas"],
        lambda r: r["nulos"] > 0 or r["vacios_trim"] > 0,
        show=0,
    )
)


💾 CSV guardado exitosamente: C:\Users\cabre\Desktop\University\Tesis\python-library-etl-master\python-library-etl-master\colombia_saludable\planes_y_evidencia\perfil_columnas.csv (107 registros, 8 columnas)
💾 CSV guardado exitosamente: C:\Users\cabre\Desktop\University\Tesis\python-library-etl-master\python-library-etl-master\colombia_saludable\planes_y_evidencia\duplicados.csv (18 registros, 3 columnas)


,tabla,tiene_pk,filas_duplicadas_excedentes
0,beneficiario,True,0
1,citas_generales,True,0
2,cotizante,True,0
3,cotizante_beneficiario,False,0
4,drogueria,True,0
5,empresa,True,0
6,empresa_cotizante,True,0
7,formulas_medicas,True,0
8,hospitalizaciones,True,0
9,ips,True,0


,tabla,columna,tipo,nulos,distintos_no_nulos,vacios_trim,minimo,maximo
63,ips,nivel,character varying,0,6,59,,
71,medico,subespecialidad,character varying,0,12,627,,
73,medico,direccion_consultorio,character varying,0,26,696,,
95,retiros,cambio_a_eps,character varying,0,5,41,,


## 4. Integridad referencial declarada y relaciones implícitas

Además de las FK físicas se prueba `id_usuario` contra cotizantes y beneficiarios.
Se distingue entre IDs ausentes, exclusivos y ambiguos (presentes en ambos tipos de persona).
Los puentes se revisan antes de asumir una única familia o empresa por afiliado.

**Funciones utilizadas en la siguiente celda**

- **Core, directamente o a través de auxiliares:** `CSV_Loader.save_csv`, `DB_Extractor.execute_query`.
- **Fuera del Core y motivo:** Las consultas SQL NOT EXISTS, EXISTS y GROUP BY comprueban relaciones declaradas e implícitas; Python configura qué relaciones y reglas del dominio se revisan.

**Funciones propias definidas o llamadas en esta celda**

| Función | Relación con el Core y motivo de conservarla |
| --- | --- |
| `guardar` | delega el CSV al Core y conserva el registro de evidencias y la presentación IPython. |
| `ident` | escapa identificadores SQL del catálogo; no calcula medidas. |
| `q` | delega el SQL a DB_Extractor.execute_query y silencia sus mensajes. |
| `tabla` | construye el nombre SQL public.tabla con identificadores escapados. |

**SQL:** las consultas de esta celda se ejecutan mediante `DB_Extractor.execute_query`, pero sus expresiones SQL pertenecen al cuaderno. Se conserva SQL para perfilado, validación o conciliación; la extracción B1 solo selecciona columnas y la transformación tabular de B2–B5 usa los transformadores Core.

**Llamadas externas directas de esta celda** (incluye los cuerpos de funciones definidos aquí; las dependencias de auxiliares llamados se explican donde se definen; `…` omite argumentos):

| Grupo | Funciones y métodos | Por qué se conservan fuera del Core |
| --- | --- | --- |
| Python estándar y contenedores | `controles.items`, `int`, `list`, `relaciones.append`, `str.join`, `usuarios.append`, `zip` | Orquestación, condiciones, iteración, construcción de SQL, parámetros y mensajes. No sustituyen una transformación tabular del Core. |
| DataFrames y matrices | `pd.DataFrame`, `q(…).iloc[…].to_dict` | Construcción de evidencia, acceso/iteración de resultados, selección de máximos y presentación de matrices. Los filtros, CSV, conversiones y deduplicación con equivalencia directa se delegan al Core. head del Core devuelve texto; no hay pivot/reindex tabular equivalente en estos transformadores. |

In [5]:
# Integridad referencial física y relaciones implícitas del negocio.


# Revisar primero las FK declaradas, incluyendo claves compuestas.
relaciones = []
for t in nombres:
    for fk in esquemas[t]["foreign_keys"]:
        pares = list(zip(fk["constrained_columns"], fk["referred_columns"]))
        no_nulos = ' AND '.join('s.' + ident(a) + ' IS NOT NULL' for a, b in pares)
        enlace = ' AND '.join('s.' + ident(a) + '=d.' + ident(b) for a, b in pares)
        h = q(
            origen,
            f'SELECT COUNT(*) AS huerfanos FROM {tabla(t)} s WHERE {no_nulos} AND NOT'
            f' EXISTS (SELECT 1 FROM {tabla(fk["referred_table"])} d WHERE {enlace})',
        ).iloc[0, 0]
        relaciones.append(
            {
                "tabla": t,
                "columnas": ','.join(fk["constrained_columns"]),
                "destino": fk["referred_table"],
                "pk_destino": ','.join(fk["referred_columns"]),
                "huerfanos": int(h),
            }
        )
guardar("integridad_fk_declaradas", pd.DataFrame(relaciones))

# Distinguir usuarios cotizantes, beneficiarios, ambiguos y sin correspondencia.
usuarios = []
for t in [
    "citas_generales",
    "urgencias",
    "hospitalizaciones",
    "remisiones",
    "formulas_medicas",
    "preexistencias",
    "pagos",
    "retiros",
]:
    datos = (
        q(
            origen,
            f"""WITH u AS (SELECT s.id_usuario,
        EXISTS(SELECT 1 FROM public.cotizante c WHERE c.cedula=s.id_usuario) AS c,
        EXISTS(SELECT 1 FROM public.beneficiario b WHERE b.id_beneficiario=s.id_usuario) AS b
        FROM {tabla(t)} s)
        SELECT COUNT(*) AS filas, COUNT(*) FILTER(WHERE c AND NOT b) AS solo_cotizante,
        COUNT(*) FILTER(WHERE b AND NOT c) AS solo_beneficiario, COUNT(*) FILTER(WHERE c AND b) AS ambos,
        COUNT(*) FILTER(WHERE NOT c AND NOT b AND id_usuario IS NOT NULL) AS sin_correspondencia,
        COUNT(*) FILTER(WHERE id_usuario IS NULL) AS id_nulo FROM u""",
        )
        .iloc[0]
        .to_dict()
    )
    usuarios.append({"tabla": t, **datos})
guardar("correspondencia_usuarios", pd.DataFrame(usuarios))

# Comprobar reglas adicionales que no están garantizadas por el DDL de origen.
controles = {
    "ids_persona_en_ambos_tipos": (
        "SELECT COUNT(*) AS valor FROM public.cotizante c JOIN public.beneficiario b ON"
        " c.cedula=b.id_beneficiario"
    ),
    "medico_ips_sin_correspondencia": (
        "SELECT COUNT(*) AS valor FROM public.medico m WHERE m.id_ips IS NOT NULL AND"
        " NOT EXISTS(SELECT 1 FROM public.ips i WHERE i.id_ips=m.id_ips)"
    ),
    "beneficiarios_con_varios_cotizantes": (
        "SELECT COUNT(*) AS valor FROM (SELECT beneficiario FROM"
        " public.cotizante_beneficiario GROUP BY beneficiario HAVING COUNT(DISTINCT"
        " cotizante)>1) x"
    ),
    "cotizantes_con_varias_empresas": (
        "SELECT COUNT(*) AS valor FROM (SELECT cotizante FROM public.empresa_cotizante"
        " GROUP BY cotizante HAVING COUNT(DISTINCT empresa)>1) x"
    ),
    "beneficiarios_sin_familia": (
        "SELECT COUNT(*) AS valor FROM public.beneficiario b WHERE NOT EXISTS(SELECT 1"
        " FROM public.cotizante_beneficiario c WHERE c.beneficiario=b.id_beneficiario)"
    ),
    "cotizantes_sin_empresa": (
        "SELECT COUNT(*) AS valor FROM public.cotizante c WHERE NOT EXISTS(SELECT 1"
        " FROM public.empresa_cotizante e WHERE e.cotizante=c.cedula)"
    ),
    "nacimientos_posteriores_afiliacion": (
        "SELECT COUNT(*) AS valor FROM public.cotizante WHERE"
        " fecha_nacimiento>fecha_afiliacion"
    ),
    "pagos_no_positivos": (
        "SELECT COUNT(*) AS valor FROM public.pagos WHERE valor_pagado<=0"
    ),
    "tarifas_negativas": (
        "SELECT COUNT(*) AS valor FROM public.servicios_pos WHERE costo<0"
    ),
    "estancias_negativas": (
        "SELECT COUNT(*) AS valor FROM public.hospitalizaciones WHERE"
        " duracion_hospitalizacion<0"
    ),
}
guardar(
    "controles_negocio",
    pd.DataFrame(
        [
            {"control": k, "valor": int(q(origen, sql).iloc[0, 0])}
            for k, sql in controles.items()
        ]
    ),
)


💾 CSV guardado exitosamente: C:\Users\cabre\Desktop\University\Tesis\python-library-etl-master\python-library-etl-master\colombia_saludable\planes_y_evidencia\integridad_fk_declaradas.csv (16 registros, 5 columnas)


,tabla,columnas,destino,pk_destino,huerfanos
0,citas_generales,id_medico,medico,cedula,0
1,cotizante,id_ips,ips,id_ips,0
2,cotizante_beneficiario,beneficiario,beneficiario,id_beneficiario,0
3,cotizante_beneficiario,cotizante,cotizante,cedula,0
4,empresa_cotizante,cotizante,cotizante,cedula,0
5,empresa_cotizante,empresa,empresa,nit,0
6,formulas_medicas,id_medico,medico,cedula,0
7,hospitalizaciones,id_medico,medico,cedula,0
8,ips_drogueria,id_drogueria,drogueria,id_drogueria,0
9,ips_drogueria,id_ips,ips,id_ips,0


💾 CSV guardado exitosamente: C:\Users\cabre\Desktop\University\Tesis\python-library-etl-master\python-library-etl-master\colombia_saludable\planes_y_evidencia\correspondencia_usuarios.csv (8 registros, 7 columnas)


,tabla,filas,solo_cotizante,solo_beneficiario,ambos,sin_correspondencia,id_nulo
0,citas_generales,3013,1353,1660,0,0,0
1,urgencias,559,256,303,0,0,0
2,hospitalizaciones,893,389,504,0,0,0
3,remisiones,16742,7281,9461,0,0,0
4,formulas_medicas,7484,3327,4157,0,0,0
5,preexistencias,4416,1100,3316,0,0,0
6,pagos,22938,22938,0,0,0,0
7,retiros,89,89,0,0,0,0


💾 CSV guardado exitosamente: C:\Users\cabre\Desktop\University\Tesis\python-library-etl-master\python-library-etl-master\colombia_saludable\planes_y_evidencia\controles_negocio.csv (10 registros, 2 columnas)


,control,valor
0,ids_persona_en_ambos_tipos,0
1,medico_ips_sin_correspondencia,0
2,beneficiarios_con_varios_cotizantes,0
3,cotizantes_con_varias_empresas,0
4,beneficiarios_sin_familia,0
5,cotizantes_sin_empresa,175
6,nacimientos_posteriores_afiliacion,0
7,pagos_no_positivos,0
8,tarifas_negativas,0
9,estancias_negativas,0


## 5. Fechas de eventos, espera y edad al evento

La diferencia disponible es **solicitud/remisión a atención**, no necesariamente asignación a atención.
Se cuenta la información incompleta y las esperas negativas sin imputar ni descartar registros.
La edad se calcula a fecha del evento, nunca con la fecha actual para hechos históricos.

**Funciones utilizadas en la siguiente celda**

- **Core, directamente o a través de auxiliares:** `CSV_Loader.save_csv`, `DB_Extractor.execute_query`.
- **Fuera del Core y motivo:** SQL EXTRACT, AGE, AVG y FILTER define el diagnóstico de edad y espera. Son consultas de perfilado del origen, no la transformación ETL de la parte B.

**Funciones propias definidas o llamadas en esta celda**

| Función | Relación con el Core y motivo de conservarla |
| --- | --- |
| `guardar` | delega el CSV al Core y conserva el registro de evidencias y la presentación IPython. |
| `ident` | escapa identificadores SQL del catálogo; no calcula medidas. |
| `q` | delega el SQL a DB_Extractor.execute_query y silencia sus mensajes. |
| `tabla` | construye el nombre SQL public.tabla con identificadores escapados. |

**SQL:** las consultas de esta celda se ejecutan mediante `DB_Extractor.execute_query`, pero sus expresiones SQL pertenecen al cuaderno. Se conserva SQL para perfilado, validación o conciliación; la extracción B1 solo selecciona columnas y la transformación tabular de B2–B5 usa los transformadores Core.

**Llamadas externas directas de esta celda** (incluye los cuerpos de funciones definidos aquí; las dependencias de auxiliares llamados se explican donde se definen; `…` omite argumentos):

| Grupo | Funciones y métodos | Por qué se conservan fuera del Core |
| --- | --- | --- |
| Python estándar y contenedores | `edades.append`, `tiempos.append` | Orquestación, condiciones, iteración, construcción de SQL, parámetros y mensajes. No sustituyen una transformación tabular del Core. |
| DataFrames y matrices | `pd.DataFrame`, `q(…).iloc[…].to_dict` | Construcción de evidencia, acceso/iteración de resultados, selección de máximos y presentación de matrices. Los filtros, CSV, conversiones y deduplicación con equivalencia directa se delegan al Core. head del Core devuelve texto; no hay pivot/reindex tabular equivalente en estos transformadores. |

In [6]:
# Diagnóstico temporal de solicitudes, atenciones y edades.


# Conservar por separado las esperas negativas y las que no pueden calcularse.
tiempos = []
edades = []
for t, fecha, hora in [
    ("citas_generales", "fecha_solicitud", "hora_solicitud"),
    ("urgencias", "fecha_solicitud", "hora_solicitud"),
    ("hospitalizaciones", "fecha_solicitud", "hora_solicitud"),
    ("remisiones", "fecha_remision", "hora_remision"),
]:
    sql = f"""WITH e AS (SELECT *, EXTRACT(EPOCH FROM ((fecha_atencion+hora_atencion)-({ident(fecha)}+{ident(hora)})))/60 AS minutos
       FROM {tabla(t)}) SELECT COUNT(*) AS filas,
       COUNT(*) FILTER(WHERE minutos IS NULL) AS espera_incompleta,
       COUNT(*) FILTER(WHERE minutos<0) AS espera_negativa,
       MIN({ident(fecha)}) AS primera_solicitud, MAX({ident(fecha)}) AS ultima_solicitud,
       MIN(fecha_atencion) AS primera_atencion, MAX(fecha_atencion) AS ultima_atencion,
       MIN(minutos) AS minimo_minutos, MAX(minutos) AS maximo_minutos,
       AVG(minutos) FILTER(WHERE minutos>=0) AS promedio_minutos_no_negativos FROM e"""
    tiempos.append({"tabla": t, **q(origen, sql).iloc[0].to_dict()})

    # Evaluar la edad al evento histórico y detectar fechas imposibles.
    sql_edad = f"""WITH p AS (SELECT cedula AS id, fecha_nacimiento FROM public.cotizante
        UNION ALL SELECT id_beneficiario, fecha_nacimiento FROM public.beneficiario)
        SELECT COUNT(*) FILTER(WHERE p.id IS NULL) AS sin_persona,
        COUNT(*) FILTER(WHERE p.fecha_nacimiento IS NULL) AS sin_nacimiento,
        COUNT(*) FILTER(WHERE s.{ident(fecha)}<p.fecha_nacimiento) AS evento_antes_nacimiento,
        COUNT(*) FILTER(WHERE EXTRACT(YEAR FROM AGE(s.{ident(fecha)},p.fecha_nacimiento))>120) AS edad_mayor_120
        FROM {tabla(t)} s LEFT JOIN p ON s.id_usuario=p.id"""
    edades.append({"tabla": t, **q(origen, sql_edad).iloc[0].to_dict()})
guardar("tiempos_eventos", pd.DataFrame(tiempos))
guardar("edades_eventos", pd.DataFrame(edades))


💾 CSV guardado exitosamente: C:\Users\cabre\Desktop\University\Tesis\python-library-etl-master\python-library-etl-master\colombia_saludable\planes_y_evidencia\tiempos_eventos.csv (4 registros, 11 columnas)


,tabla,filas,espera_incompleta,espera_negativa,primera_solicitud,ultima_solicitud,primera_atencion,ultima_atencion,minimo_minutos,maximo_minutos,promedio_minutos_no_negativos
0,citas_generales,3013,0,0,2006-01-09,2008-11-09,2006-01-12,2008-11-14,782.0,8636.0,3863.779622
1,urgencias,559,0,0,2006-01-17,2008-11-08,2006-01-17,2008-11-08,20.0,45.0,32.822898
2,hospitalizaciones,893,0,0,2006-01-17,2008-11-08,2006-01-17,2008-11-08,20.0,45.0,32.416573
3,remisiones,16742,0,497,2006-01-13,2008-11-09,2006-01-17,2008-12-09,-120.0,86400.0,17571.079902


💾 CSV guardado exitosamente: C:\Users\cabre\Desktop\University\Tesis\python-library-etl-master\python-library-etl-master\colombia_saludable\planes_y_evidencia\edades_eventos.csv (4 registros, 5 columnas)


,tabla,sin_persona,sin_nacimiento,evento_antes_nacimiento,edad_mayor_120
0,citas_generales,0,0,0,0
1,urgencias,0,0,0,0
2,hospitalizaciones,0,0,0,0
3,remisiones,0,0,0,0


## 6. Catálogos relevantes para el taller

Se preservan etiquetas originales (incluida `estracto`). Las distribuciones ayudan a definir normalización,
no justifican inventar categorías. Departamento/municipio pertenecen a la IPS: no prueban residencia del paciente.
Se muestran catálogos clínicos y demográficos agregados, nunca nombres, cédulas o direcciones.

**Funciones utilizadas en la siguiente celda**

- **Core, directamente o a través de auxiliares:** `CSV_Loader.save_csv`, `DB_Extractor.execute_query`, `DataSelect.filter_equal`, `DataSelect.filter_in_list`.
- **Fuera del Core y motivo:** SQL GROUP BY calcula distribuciones y catálogos en origen; Python reúne los registros. Los filtros tabulares de presentación se delegan al Core.

**Funciones propias definidas o llamadas en esta celda**

| Función | Relación con el Core y motivo de conservarla |
| --- | --- |
| `guardar` | delega el CSV al Core y conserva el registro de evidencias y la presentación IPython. |
| `ident` | escapa identificadores SQL del catálogo; no calcula medidas. |
| `q` | delega el SQL a DB_Extractor.execute_query y silencia sus mensajes. |
| `tabla` | construye el nombre SQL public.tabla con identificadores escapados. |

**SQL:** las consultas de esta celda se ejecutan mediante `DB_Extractor.execute_query`, pero sus expresiones SQL pertenecen al cuaderno. Se conserva SQL para perfilado, validación o conciliación; la extracción B1 solo selecciona columnas y la transformación tabular de B2–B5 usa los transformadores Core.

**Llamadas externas directas de esta celda** (incluye los cuerpos de funciones definidos aquí; las dependencias de auxiliares llamados se explican donde se definen; `…` omite argumentos):

| Grupo | Funciones y métodos | Por qué se conservan fuera del Core |
| --- | --- | --- |
| Python estándar y contenedores | `categorias.extend`, `diagnosticos.extend` | Orquestación, condiciones, iteración, construcción de SQL, parámetros y mensajes. No sustituyen una transformación tabular del Core. |
| DataFrames y matrices | `d.to_dict`, `pd.DataFrame` | Construcción de evidencia, acceso/iteración de resultados, selección de máximos y presentación de matrices. Los filtros, CSV, conversiones y deduplicación con equivalencia directa se delegan al Core. head del Core devuelve texto; no hay pivot/reindex tabular equivalente en estos transformadores. |
| Dibujo e IPython | `display` | Renderizado, ejes, leyendas, imágenes y visualización del notebook; el Core no ofrece renderizadores Python de gráficos ni display HTML. |

In [7]:
# Catálogos y distribuciones que orientan el diseño de la bodega.


# Conservar las etiquetas originales antes de definir reglas de normalización.
categorias = []
for t, c in [
    ("cotizante", "sexo"),
    ("beneficiario", "sexo"),
    ("cotizante", "tipo_cotizante"),
    ("cotizante", "estracto"),
    ("cotizante", "proviene_otra_eps"),
    ("cotizante", "nivel_escolaridad"),
    ("ips", "departamento"),
    ("ips", "municipio"),
    ("ips", "tipo_ips"),
    ("ips", "nivel"),
    ("retiros", "cambio_a_eps"),
    ("preexistencias", "familiar"),
    ("preexistencias", "enfermedad"),
]:
    d = q(
        origen,
        f'SELECT {ident(c)} AS valor, COUNT(*) AS filas FROM {tabla(t)} GROUP BY 1'
        ' ORDER BY 2 DESC, 1',
    )
    categorias.extend({"tabla": t, "columna": c, **r} for r in d.to_dict("records"))
guardar("distribuciones_catalogos", pd.DataFrame(categorias), mostrar=False)

# Revisar el catálogo de servicios y sus tarifas sin equipararlas a costos reales.
servicios = q(
    origen,
    "SELECT id_servicio_pos, descripcion, costo FROM public.servicios_pos ORDER BY"
    " id_servicio_pos",
)
guardar("catalogo_servicios", servicios)

# Contar los textos diagnósticos antes de definir una clasificación explícita.
diagnosticos = []
for t in ["citas_generales", "urgencias", "hospitalizaciones", "remisiones"]:
    d = q(
        origen,
        f'SELECT diagnostico, COUNT(*) AS filas FROM {tabla(t)} GROUP BY 1 ORDER BY 2'
        ' DESC, 1',
    )
    diagnosticos.extend({"tabla": t, **r} for r in d.to_dict("records"))
guardar("distribucion_diagnosticos", pd.DataFrame(diagnosticos), mostrar=False)
display(
    S.filter_in_list(
        S.filter_equal(
            evidencias["distribuciones_catalogos"], "tabla", "cotizante", show=0
        ),
        "columna",
        ["sexo", "tipo_cotizante", "estracto", "proviene_otra_eps"],
        show=0,
    )
)


💾 CSV guardado exitosamente: C:\Users\cabre\Desktop\University\Tesis\python-library-etl-master\python-library-etl-master\colombia_saludable\planes_y_evidencia\distribuciones_catalogos.csv (77 registros, 4 columnas)
💾 CSV guardado exitosamente: C:\Users\cabre\Desktop\University\Tesis\python-library-etl-master\python-library-etl-master\colombia_saludable\planes_y_evidencia\catalogo_servicios.csv (67 registros, 3 columnas)


,id_servicio_pos,descripcion,costo
0,1234,Consulta General,15850
1,1241,Cirugía Drenaje Tejido Celular Subcutáneo,50000
2,1248,Cirugía Amigdalectomía,543900
3,1255,Examen Esofagogastroduodenoscopia,58000
4,1262,Examen Biopsia,50500
...,...,...,...
62,1668,Tratamiento HAART,625000
63,1675,Examen de Embarazo,20000
64,1682,Tratamiento Control Embarazo,35000
65,1689,Cirugía Cesárea,765000


💾 CSV guardado exitosamente: C:\Users\cabre\Desktop\University\Tesis\python-library-etl-master\python-library-etl-master\colombia_saludable\planes_y_evidencia\distribucion_diagnosticos.csv (109 registros, 3 columnas)


,tabla,columna,valor,filas
0,cotizante,sexo,M,563
1,cotizante,sexo,F,537
4,cotizante,tipo_cotizante,Empleado,765
5,cotizante,tipo_cotizante,Jubilado,294
6,cotizante,tipo_cotizante,Independiente,41
7,cotizante,estracto,3,231
8,cotizante,estracto,4,224
9,cotizante,estracto,2,221
10,cotizante,estracto,5,214
11,cotizante,estracto,1,210


## 7. Medicamentos y cobertura de tarifas

Se comprueba la lista delimitada por `;`, los tokens vacíos/repetidos y su correspondencia con `servicios_pos`.
Un precio de catálogo no equivale al costo real de una atención ni certifica unidades entregadas.
Las preexistencias describen familiares: no se tratarán automáticamente como diagnósticos del afiliado.

**Funciones utilizadas en la siguiente celda**

- **Core, directamente o a través de auxiliares:** `CSV_Loader.save_csv`, `DB_Extractor.execute_query`.
- **Fuera del Core y motivo:** regexp_split_to_table, BTRIM, GROUP BY y los joins SQL diagnostican tokens y cobertura del catálogo sin construir aún las tablas ETL.

**Funciones propias definidas o llamadas en esta celda**

| Función | Relación con el Core y motivo de conservarla |
| --- | --- |
| `guardar` | delega el CSV al Core y conserva el registro de evidencias y la presentación IPython. |
| `q` | delega el SQL a DB_Extractor.execute_query y silencia sus mensajes. |

**SQL:** las consultas de esta celda se ejecutan mediante `DB_Extractor.execute_query`, pero sus expresiones SQL pertenecen al cuaderno. Se conserva SQL para perfilado, validación o conciliación; la extracción B1 solo selecciona columnas y la transformación tabular de B2–B5 usa los transformadores Core.

**Llamadas externas directas de esta celda:** ninguna adicional; las llamadas propias delegan en los auxiliares documentados y sus dependencias se explican donde se definen.

In [8]:
# Calidad de los códigos de medicamento y cobertura de tarifas.


# Separar los códigos de cada receta sin atribuirles dosis ni cantidades.
tokens_cte = """WITH tokens AS (SELECT f.codigo_formula, BTRIM(x.token) AS codigo
FROM public.formulas_medicas f
CROSS JOIN LATERAL regexp_split_to_table(COALESCE(f.medicamentos_recetados,''), ';') AS x(token)) """
guardar(
    "calidad_medicamentos",
    q(
        origen,
        tokens_cte + """SELECT COUNT(*) AS tokens,
COUNT(*) FILTER(WHERE codigo='') AS tokens_vacios, COUNT(DISTINCT codigo) FILTER(WHERE codigo<>'') AS codigos_distintos,
COUNT(*) FILTER(WHERE codigo<>'' AND NOT EXISTS(SELECT 1 FROM public.servicios_pos s WHERE s.id_servicio_pos=t.codigo)) AS tokens_sin_catalogo,
(SELECT COUNT(*) FROM (SELECT codigo_formula,codigo FROM tokens WHERE codigo<>'' GROUP BY 1,2 HAVING COUNT(*)>1) d) AS pares_formula_codigo_repetidos
FROM tokens t""",
    ),
)
guardar(
    "catalogo_medicamentos_observados",
    q(
        origen,
        tokens_cte + """SELECT t.codigo, s.descripcion, s.costo,
COUNT(*) AS apariciones, COUNT(DISTINCT t.codigo_formula) AS formulas
FROM tokens t LEFT JOIN public.servicios_pos s ON s.id_servicio_pos=t.codigo
WHERE t.codigo<>'' GROUP BY 1,2,3 ORDER BY formulas DESC, t.codigo""",
    ),
)
guardar(
    "cobertura_tarifas",
    q(
        origen,
        """SELECT COUNT(*) AS remisiones,
COUNT(*) FILTER(WHERE r.servicio_pos IS NULL) AS sin_servicio,
COUNT(*) FILTER(WHERE s.id_servicio_pos IS NULL AND r.servicio_pos IS NOT NULL) AS servicio_sin_catalogo,
COUNT(*) FILTER(WHERE s.costo IS NULL) AS sin_tarifa,
COUNT(*) FILTER(WHERE LOWER(s.descripcion) LIKE '%cirug%') AS remisiones_cirugia_por_texto
FROM public.remisiones r LEFT JOIN public.servicios_pos s ON s.id_servicio_pos=r.servicio_pos""",
    ),
)


💾 CSV guardado exitosamente: C:\Users\cabre\Desktop\University\Tesis\python-library-etl-master\python-library-etl-master\colombia_saludable\planes_y_evidencia\calidad_medicamentos.csv (1 registros, 5 columnas)


,tokens,tokens_vacios,codigos_distintos,tokens_sin_catalogo,pares_formula_codigo_repetidos
0,15770,0,166,15770,0


💾 CSV guardado exitosamente: C:\Users\cabre\Desktop\University\Tesis\python-library-etl-master\python-library-etl-master\colombia_saludable\planes_y_evidencia\catalogo_medicamentos_observados.csv (166 registros, 5 columnas)


,codigo,descripcion,costo,apariciones,formulas
0,2349,None,None,895,895
1,2356,None,None,861,861
2,1047,None,None,850,850
3,2503,None,None,593,593
4,1257,None,None,441,441
...,...,...,...,...,...
161,1992,None,None,3,3
162,2013,None,None,3,3
163,2020,None,None,3,3
164,1964,None,None,2,2


💾 CSV guardado exitosamente: C:\Users\cabre\Desktop\University\Tesis\python-library-etl-master\python-library-etl-master\colombia_saludable\planes_y_evidencia\cobertura_tarifas.csv (1 registros, 5 columnas)


,remisiones,sin_servicio,servicio_sin_catalogo,sin_tarifa,remisiones_cirugia_por_texto
0,16742,0,0,0,960


## 8. Interpretación para el diseño

Las verificaciones anteriores sustentan las decisiones; los planes no suponen que todos los campos del relato
del taller estén implementados en el origen. Los resultados siguientes se recalculan al ejecutar el diagnóstico.

**Funciones utilizadas en la siguiente celda**

- **Core, directamente o a través de auxiliares:** ninguna; esta celda interpreta o documenta resultados ya obtenidos.
- **Fuera del Core y motivo:** pandas set_index/loc/iloc y sum acceden a agregados ya calculados; IPython/Markdown presenta conclusiones del diseño. No son nuevas transformaciones de hechos.

**Llamadas externas directas de esta celda** (incluye los cuerpos de funciones definidos aquí; las dependencias de auxiliares llamados se explican donde se definen; `…` omite argumentos):

| Grupo | Funciones y métodos | Por qué se conservan fuera del Core |
| --- | --- | --- |
| Python estándar y contenedores | `int` | Orquestación, condiciones, iteración, construcción de SQL, parámetros y mensajes. No sustituyen una transformación tabular del Core. |
| DataFrames y matrices | `evidencias[…].set_index`, `evidencias[…][…].sum` | Construcción de evidencia, acceso/iteración de resultados, selección de máximos y presentación de matrices. Los filtros, CSV, conversiones y deduplicación con equivalencia directa se delegan al Core. head del Core devuelve texto; no hay pivot/reindex tabular equivalente en estos transformadores. |
| Dibujo e IPython | `Markdown`, `display` | Renderizado, ejes, leyendas, imágenes y visualización del notebook; el Core no ofrece renderizadores Python de gráficos ni display HTML. |

In [9]:
# Interpretación de la evidencia para decidir el modelo dimensional.


# Recuperar los agregados ya calculados para redactar las decisiones de diseño.
v = evidencias["volumen_origen"].set_index("tabla")["filas"]
negativas = int(
    evidencias["tiempos_eventos"]
    .set_index("tabla")
    .loc["remisiones", "espera_negativa"]
)
med = evidencias["calidad_medicamentos"].iloc[0]
display(Markdown(f"""
- **Personas:** {v['cotizante']:,} cotizantes y {v['beneficiario']:,} beneficiarios. No unir eventos solo a cotizantes.
- **Integridad:** {int(evidencias['integridad_fk_declaradas']['huerfanos'].sum())} huérfanos en las FK declaradas.
  Revisar también la evidencia de relaciones implícitas, ausentes del DDL original.
- **Espera:** {negativas:,} remisiones con diferencia negativa. Conservar el evento y marcar la espera como no válida;
  no usar valor absoluto ni convertir el error en cero.
- **Medicamentos:** {int(med['codigos_distintos']):,} códigos distintos y {int(med['tokens_sin_catalogo']):,} tokens sin catálogo.
  Frecuencias y coformulación son posibles por código; nombres, dosis y costo de receta requieren otra fuente.
- **Pagos:** aportes recibidos; no representan costos asistenciales. Las tarifas de `servicios_pos` solo permiten una
  estimación de servicios identificados en remisiones, sin asumir una factura ni un precio histórico.
- **Retiros:** `cambio_a_eps` vacío no demuestra mora. Sin vencimiento, período cotizado ni causa de retiro,
  no se pueden responder de forma exacta las preguntas sobre pago tardío.
- **Ocupación y geografía:** eventos por médico/IPS no miden capacidad ocupada; faltan camas y agendas.
  Municipio/departamento de la IPS no son domicilio del paciente ni una regional administrativa definida.
- **Core:** ofrece extracción, transformaciones tabulares, carga y analítica; no genera
  gráficos Python. El Core actual sí ofrece `LoadSchemaSpec`/`persist_schema` para PK/FK/índices;
  este cuaderno conserva la carga con `DB_Loader` y relaciones lógicas del contrato publicado.
"""))



- **Personas:** 1,100 cotizantes y 3,316 beneficiarios. No unir eventos solo a cotizantes.
- **Integridad:** 0 huérfanos en las FK declaradas.
  Revisar también la evidencia de relaciones implícitas, ausentes del DDL original.
- **Espera:** 497 remisiones con diferencia negativa. Conservar el evento y marcar la espera como no válida;
  no usar valor absoluto ni convertir el error en cero.
- **Medicamentos:** 166 códigos distintos y 15,770 tokens sin catálogo.
  Frecuencias y coformulación son posibles por código; nombres, dosis y costo de receta requieren otra fuente.
- **Pagos:** aportes recibidos; no representan costos asistenciales. Las tarifas de `servicios_pos` solo permiten una
  estimación de servicios identificados en remisiones, sin asumir una factura ni un precio histórico.
- **Retiros:** `cambio_a_eps` vacío no demuestra mora. Sin vencimiento, período cotizado ni causa de retiro,
  no se pueden responder de forma exacta las preguntas sobre pago tardío.
- **Ocupación y geografía:** eventos por médico/IPS no miden capacidad ocupada; faltan camas y agendas.
  Municipio/departamento de la IPS no son domicilio del paciente ni una regional administrativa definida.
- **Core:** ofrece extracción, transformaciones tabulares, carga y analítica; no genera
  gráficos Python. El Core actual sí ofrece `LoadSchemaSpec`/`persist_schema` para PK/FK/índices;
  este cuaderno conserva la carga con `DB_Loader` y relaciones lógicas del contrato publicado.


## 9. Cierre, reproducibilidad y alcance

El catálogo final se contrasta con el inicial y se cierran los pools. La protección de lectura se aplica a todas
las consultas del diagnóstico. La igualdad de catálogos confirma ausencia de cambios estructurales observados;
no pretende demostrar que ningún otro proceso haya modificado datos.

Los siguientes archivos desarrollan las decisiones a partir de esta evidencia:

- `../planes_y_evidencia/analisis_origen.md`
- `../planes_y_evidencia/plan_etl_core.md`
- `../planes_y_evidencia/plan_answers.md`

**Este cierre corresponde solo al diagnóstico. La implementación ETL está en la parte B.**

**Funciones utilizadas en la siguiente celda**

- **Core, directamente o a través de auxiliares:** `CSV_Loader.save_csv`, `DB_Extractor.close_connection`, `DB_Extractor.execute_query`, `DB_Extractor.get_table_names`.
- **Fuera del Core y motivo:** DataFrame.equals compara catálogos; datetime/version, JSON, Path.write_text y Python registran reproducibilidad. El cierre de conexiones usa el Core.

**Funciones propias definidas o llamadas en esta celda**

| Función | Relación con el Core y motivo de conservarla |
| --- | --- |
| `guardar` | delega el CSV al Core y conserva el registro de evidencias y la presentación IPython. |
| `q` | delega el SQL a DB_Extractor.execute_query y silencia sus mensajes. |

**Llamadas externas directas de esta celda** (incluye los cuerpos de funciones definidos aquí; las dependencias de auxiliares llamados se explican donde se definen; `…` omite argumentos):

| Grupo | Funciones y métodos | Por qué se conservan fuera del Core |
| --- | --- | --- |
| Python estándar y contenedores | `cierre.append`, `extractores.items`, `extractores.values`, `len`, `print`, `sorted` | Orquestación, condiciones, iteración, construcción de SQL, parámetros y mensajes. No sustituyen una transformación tabular del Core. |
| Archivos, configuración y reproducibilidad | `ZoneInfo`, `datetime.now`, `datetime.now(…).isoformat`, `json.dumps`, `platform.python_version`, `ruta.write_text`, `version` | Rutas, entorno, manifiestos, fechas de ejecución, versiones o hashes específicos de este proceso; el Core no publica estos manifiestos del taller. |
| DataFrames y matrices | `actual.equals`, `pd.DataFrame` | Construcción de evidencia, acceso/iteración de resultados, selección de máximos y presentación de matrices. Los filtros, CSV, conversiones y deduplicación con equivalencia directa se delegan al Core. head del Core devuelve texto; no hay pivot/reindex tabular equivalente en estos transformadores. |

In [10]:
# Cierre del diagnóstico y registro de reproducibilidad.

try:

    # Contrastar los catálogos finales con la fotografía inicial de cada base.
    cierre = []
    for rol, ex in extractores.items():
        actual = q(ex, catalogo_sql)
        igual = actual.equals(inventarios_iniciales[rol])
        cierre.append(
            {"rol": rol, "catalogo_sin_cambios": igual, "objetos_finales": len(actual)}
        )
        if rol == "bodega":
            guardar("catalogo_bodega_final", actual, mostrar=False)
        assert (
            igual
        ), "El catálogo cambió durante el diagnóstico; revisar actividad externa."
    guardar("comprobacion_final", pd.DataFrame(cierre))

    # Documentar paquetes, alcance y archivos; cada consulta tiene su transacción.
    manifiesto = {
        "inicio_bogota": INICIO,
        "fin_bogota": datetime.now(ZoneInfo("America/Bogota")).isoformat(),
        "modo": "diagnostico_solo_lectura_sin_etl",
        "python": platform.python_version(),
        "paquetes": {
            p: version(p)
            for p in [
                "pandas",
                "SQLAlchemy",
                "psycopg2-binary",
                "python-dotenv",
                "nbformat",
                "nbclient",
                "ipykernel",
            ]
        },
        "tablas_origen_public": len(nombres),
        "tablas_bodega_public": len(bodega.get_table_names()),
        "archivos_csv": sorted(f"{k}.csv" for k in evidencias),
        "consistencia": (
            "Una transacción REPEATABLE READ por consulta; no hay snapshot global del"
            " diagnóstico."
        ),
    }
    (EVIDENCIA / "ejecucion_diagnostico.json").write_text(
        json.dumps(manifiesto, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    print("Evidencia guardada. No se ejecutó el ETL.")
finally:

    # Cerrar conexiones incluso si una comprobación o exportación falla.
    for ex in extractores.values():
        ex.close_connection()


💾 CSV guardado exitosamente: C:\Users\cabre\Desktop\University\Tesis\python-library-etl-master\python-library-etl-master\colombia_saludable\planes_y_evidencia\catalogo_bodega_final.csv (0 registros, 3 columnas)
💾 CSV guardado exitosamente: C:\Users\cabre\Desktop\University\Tesis\python-library-etl-master\python-library-etl-master\colombia_saludable\planes_y_evidencia\comprobacion_final.csv (2 registros, 3 columnas)


,rol,catalogo_sin_cambios,objetos_finales
0,origen,True,18
1,bodega,True,0


📊 Tablas encontradas: 0
Evidencia guardada. No se ejecutó el ETL.
🔒 Conexión cerrada.
🔒 Conexión cerrada.


# Parte B. ETL Core: construcción, carga inicial y conciliación

Implementación del [plan ETL](../planes_y_evidencia/plan_etl_core.md). Todas las extracciones,
uniones, proyecciones, derivaciones y escrituras de datos utilizan las funciones de `etl/`.
Python aporta orquestación, expresiones de fila, mapeos y validación. No se modifica el Core.

Por defecto, **ejecutar todas las celdas prepara y valida en memoria, sin cargar**. Para cargar una bodega vacía,
usar desde la raíz `python colombia_saludable/planes_y_evidencia/ejecutar_etl.py --cargar`.
La opción se transmite al kernel con `COLOMBIA_SALUDABLE_CARGAR=1`; no contiene credenciales.
La prevalidación bloquea cualquier carga si existe un objeto de usuario en destino.

La ruta `DB_Loader` usada aquí no crea PK/FK/índices físicos y no es atómica entre tablas.
El Core actual ofrece `LoadSchemaSpec` y `persist_schema` para esquemas con restricciones;
adoptarlos cambiaría el contrato lógico publicado y requiere una migración separada.
Las referencias opcionales se validan antes de cargar y después con SELECT, fuera del mapa del loader, conservando sus nulos.
No se imprimen personas ni filas de hechos. La evidencia contiene catálogos públicos, metadatos y agregados.

**Funciones utilizadas en la siguiente celda**

- **Core, directamente o a través de auxiliares:** `BasicsTransformOperations.add_new_column`, `DB_Extractor.close_connection`, `DB_Extractor.connect`, `DB_Extractor.execute_query`, `DataSelect.filter_in_list`, `DataSelect.filter_is_null`, `DataSelect.select_columns`, `HeaderOperations.rename_columns`, `TransformOperations.drop_duplicates`, `TransformOperations.left_join`, `etl.DB_Extractor`, `etl.schema.missing (alias nulo)`.
- **Fuera del Core y motivo:** unicodedata, datetime/date/time, Decimal y hashlib proporcionan expresiones escalares y hashes. SQLAlchemy y el driver configuran sesiones; os/uuid/Path/JSON mantienen estado. Las funciones propias se detallan abajo: delegar su aplicación al Core no convierte sus reglas en funciones del Core.

**Funciones propias definidas o llamadas en esta celda**

| Función | Relación con el Core y motivo de conservarla |
| --- | --- |
| `clave` | define el tratamiento de claves nulas y espacios como expresión de fila. |
| `conectar_etl` | conecta mediante DB_Extractor y añade configuración de sesión del driver. |
| `configurar` | establece aislamiento y modo de lectura de la conexión mediante el driver. |
| `derivar` | es un envoltorio de BasicsTransformOperations.add_new_column; el callable expresa la regla de negocio. |
| `dinero` | aplica la cuantización Decimal a dos decimales; las conversiones estrictas del Core no sustituyen esa política de redondeo. |
| `edad` | define años cumplidos entre nacimiento y evento como expresión de fila del ETL. |
| `exigir` | comprueba condiciones y emite errores de negocio del taller. |
| `fecha` | interpreta valores de fecha del origen como auxiliar escalar de las derivaciones. |
| `fecha_id` | define la clave de calendario YYYYMMDD del modelo. |
| `grupo_edad` | define los intervalos y la categoría desconocido del taller. |
| `huella_core` | calcula SHA-256 de archivos Python; no es una operación tabular del Core. |
| `huella_frame` | calcula SHA-256 de una serialización ordenada para comprobar reproducibilidad. |
| `limitar` | configura timeouts mediante SQLAlchemy; no accede a datos de negocio. |
| `limpiar` | define la regla NFC, espacios, casefold y desconocidos; se suministra como callable a los transformadores Core. |
| `minutos` | define diferencias y redondeo Decimal de seis posiciones para las esperas. |
| `proyectar` | es un envoltorio de DataSelect.select_columns. |
| `q` | delega el SQL a DB_Extractor.execute_query y silencia sus mensajes. |
| `registrar_estado` | escribe el manifiesto específico de este proceso en JSON. |
| `renombrar` | es un envoltorio de HeaderOperations.rename_columns. |
| `sello` | combina fecha y hora locales; el calendario Core no proporciona este auxiliar escalar. |
| `unico` | delega selección, nulos y deduplicación al Core y añade los mensajes/valores inválidos del contrato. |
| `unir_izquierda` | delega a TransformOperations.left_join y añade comprobaciones de cardinalidad. |

**SQL:** las consultas de esta celda se ejecutan mediante `DB_Extractor.execute_query`, pero sus expresiones SQL pertenecen al cuaderno. Se conserva SQL para perfilado, validación o conciliación; la extracción B1 solo selecciona columnas y la transformación tabular de B2–B5 usa los transformadores Core.

**Llamadas externas directas de esta celda** (incluye los cuerpos de funciones definidos aquí; las dependencias de auxiliares llamados se explican donde se definen; `…` omite argumentos):

| Grupo | Funciones y métodos | Por qué se conservan fuera del Core |
| --- | --- | --- |
| Python estándar y contenedores | `RuntimeError`, `ValueError`, `all`, `int`, `isinstance`, `len`, `p.relative_to(…).as_posix(…).encode`, `print`, `sorted`, `str`, `str(…).strip`, `str.join`, `str.join(…).encode` | Orquestación, condiciones, iteración, construcción de SQL, parámetros y mensajes. No sustituyen una transformación tabular del Core. |
| Archivos, configuración y reproducibilidad | `INTENTOS.mkdir`, `SALIDA_ETL.mkdir`, `ZoneInfo`, `contextlib.redirect_stdout`, `datetime.combine`, `datetime.now`, `datetime.now(…).isoformat`, `datetime.now(…).strftime`, `h.hexdigest`, `h.update`, `hashlib.sha256`, `hashlib.sha256(…).hexdigest`, `io.StringIO`, `json.dumps`, `os.environ.get`, `p.read_bytes`, `p.relative_to`, `p.relative_to(…).as_posix`, `ruta.rglob`, `ruta.write_text`, `uuid.uuid4` | Rutas, entorno, manifiestos, fechas de ejecución, versiones o hashes específicos de este proceso; el Core no publica estos manifiestos del taller. |
| Sesiones SQLAlchemy y driver | `conn.exec_driver_sql`, `dbapi.set_session`, `event.listens_for`, `ex.engine.dispose` | Modo de lectura, aislamiento y tiempos límite añadidos a DB_Extractor; no son cargas de datos por fuera del Core. |
| DataFrames y matrices | `frame.to_dict` | Construcción de evidencia, acceso/iteración de resultados, selección de máximos y presentación de matrices. Los filtros, CSV, conversiones y deduplicación con equivalencia directa se delegan al Core. head del Core devuelve texto; no hay pivot/reindex tabular equivalente en estos transformadores. |
| Expresiones escalares del dominio | `Decimal`, `Decimal(…).quantize`, `date.fromisoformat`, `f.strftime`, `expresión Decimal.quantize`, `str.join(…).casefold`, `time.fromisoformat`, `unicodedata.normalize`, `unicodedata.normalize(…).split`, `valor.date` | Normalización Unicode, fechas, duraciones, claves y redondeo Decimal de las reglas explícitas. Se aplican con callables Core cuando corresponden a columnas; no son funciones propias del Core. |

In [11]:
# Preparación de la carga y funciones de negocio del ETL.

import os
import hashlib
import unicodedata
import uuid
import warnings
from datetime import date, time, timedelta
from decimal import Decimal, ROUND_HALF_UP
from sqlalchemy import types as sqltypes
from etl import (
    DB_Loader,
    BasicsTransformOperations as B,
    TransformOperations as T,
    HeaderOperations as H,
    ConvertOperations as C,
    DataSelect as S,
    DataExpresion as X,
    DateTime as CalendarioCore,
)

# Mantener la preparación sin escrituras salvo activación explícita de la carga.
EJECUTAR_CARGA = os.environ.get("COLOMBIA_SALUDABLE_CARGAR", "0") == "1"
# El destino autorizado procede del .env actualizado por el usuario.
DESTINO_AUTORIZADO = cfg["DB_NAME_WAREHOUSE"]
SALIDA_ETL = EVIDENCIA / "carga_core"
SALIDA_ETL.mkdir(exist_ok=True)

# Separar los intentos y registrar su avance sin sustituir una carga exitosa.
RUN_ID = (
    datetime.now(ZoneInfo("America/Bogota")).strftime("%Y%m%dT%H%M%S")
    + "_"
    + uuid.uuid4().hex[:8]
)
INTENTOS = SALIDA_ETL / "intentos"
INTENTOS.mkdir(exist_ok=True)
estado_run = {
    "run_id": RUN_ID,
    "inicio_bogota": datetime.now(ZoneInfo("America/Bogota")).isoformat(),
    "modo": "carga" if EJECUTAR_CARGA else "preparacion",
    "estado": "prevalidacion",
    "tablas_cargadas": [],
    "destino": DESTINO_AUTORIZADO,
}


def registrar_estado():
    # Guardar el avance del intento para poder identificar una carga incompleta.
    (INTENTOS / f"{RUN_ID}.json").write_text(
        json.dumps(estado_run, ensure_ascii=False, indent=2), encoding="utf8"
    )


def exigir(condicion, mensaje):
    # Detener esta etapa si no se cumple una condición del contrato de datos.
    if not condicion:
        raise ValueError(mensaje)


def limpiar(valor):
    # Normalizar Unicode, espacios y mayúsculas sin inventar valores desconocidos.
    if nulo(valor):
        return None
    texto = ' '.join(unicodedata.normalize('NFC', str(valor)).split()).casefold()
    return None if texto in ('', 'n/a') else texto


def clave(valor):
    # Conservar los nulos reales y quitar solo espacios externos de la clave.
    return None if nulo(valor) else str(valor).strip()


def fecha(valor):
    # Obtener la fecha de calendario sin conservar una posible hora del origen.
    if nulo(valor):
        return None
    return (
        valor.date()
        if isinstance(valor, datetime)
        else date.fromisoformat(str(valor)[:10])
    )


def fecha_id(valor):
    # Usar YYYYMMDD como clave determinista, independiente del orden de las filas.
    f = fecha(valor)
    return f.strftime('%Y%m%d') if f else None


def edad(nacimiento, evento):
    # Calcular años cumplidos al evento, descontando si aún no ocurrió el cumpleaños.
    a, b = fecha(nacimiento), fecha(evento)
    return (
        None
        if a is None or b is None
        else b.year - a.year - ((b.month, b.day) < (a.month, a.day))
    )


def grupo_edad(valor):
    # Aplicar los intervalos del taller y separar las edades desconocidas.
    if nulo(valor) or valor < 0:
        return 'desconocido'
    for limite, etiqueta in [
        (4, '0-4'),
        (14, '5-14'),
        (24, '15-24'),
        (44, '25-44'),
        (64, '45-64'),
    ]:
        if valor <= limite:
            return etiqueta
    return '65+'


def dinero(valor):
    # Mantener aritmética Decimal y fijar la escala monetaria en dos decimales.
    return None if nulo(valor) else Decimal(str(valor)).quantize(Decimal('0.01'))


def sello(f, h):
    # Combinar fecha y hora locales solo cuando ambos componentes existen.
    if nulo(f) or nulo(h):
        return None
    return datetime.combine(
        fecha(f), h if isinstance(h, time) else time.fromisoformat(str(h))
    )


def minutos(inicio, fin):
    # Conservar el signo de la espera y redondear a seis decimales con Decimal.
    if nulo(inicio) or nulo(fin):
        return None
    d = fin - inicio
    return (Decimal(d.days * 86400 + d.seconds) / Decimal(60)).quantize(
        Decimal('0.000001'), rounding=ROUND_HALF_UP
    )


def derivar(frame, nombre, funcion):
    # Aplicar la expresión de negocio a cada fila mediante el transformador Core.
    return B.add_new_column(frame, nombre, funcion, show=0)


def proyectar(frame, *columnas):
    # Seleccionar y ordenar únicamente las columnas solicitadas con el Core.
    return S.select_columns(frame, *columnas, show=0)


def renombrar(frame, mapa):
    # Delegar el cambio de nombres de columnas al Core.
    return H.rename_columns(frame, mapa, show=0)


def unico(frame, columnas, etiqueta):
    # Validar la clave completa: sin nulos, falsas cadenas nulas ni duplicados.
    claves = proyectar(frame, *columnas)
    exigir(
        all(S.filter_is_null(claves, c, show=0).empty for c in columnas),
        'Clave ausente: ' + etiqueta,
    )
    # Las cadenas que aparentan nulos son una regla de negocio adicional.
    exigir(
        all(
            not S.filter_in_list(claves, c, ['', 'None', 'nan', '<NA>'], show=0).shape[
                0
            ]
            for c in columnas
        ),
        'Clave ausente: ' + etiqueta,
    )
    exigir(
        len(claves) == len(T.drop_duplicates(claves, subset=columnas, show=0)),
        'Clave no única: ' + etiqueta,
    )


def unir_izquierda(izquierda, derecha, on, etiqueta):
    # Exigir una clave única a la derecha para que el cruce no multiplique filas.
    derecha_cols = (
        [on[1]] if isinstance(on, tuple) else ([on] if isinstance(on, str) else on)
    )
    unico(derecha, derecha_cols, etiqueta)
    salida = T.left_join(izquierda, derecha, on=on, show=0)
    exigir(len(salida) == len(izquierda), 'Join multiplicó filas: ' + etiqueta)
    return salida


def huella_frame(frame):
    # Huella global de validación; nunca se guardan hashes por persona.
    # Ordenar la serialización para que el hash no dependa del orden de extracción.
    filas = sorted(
        json.dumps(r, sort_keys=True, default=str, ensure_ascii=False)
        for r in frame.to_dict('records')
    )
    return hashlib.sha256(('\n'.join(filas)).encode('utf8')).hexdigest()


def huella_core():
    # Incluir rutas y contenido para detectar cambios en los archivos del Core.
    h = hashlib.sha256()
    for p in sorted((ROOT / 'etl').rglob('*.py')):
        h.update(p.relative_to(ROOT).as_posix().encode())
        h.update(p.read_bytes())
    return h.hexdigest()


def conectar_etl(base_datos, solo_lectura):
    # Abrir con el extractor Core y configurar protección/timeouts en cada conexión.
    ex = DB_Extractor(
        db_type=cfg['DB_TYPE'],
        host=cfg['DB_HOST'],
        port=int(cfg['DB_PORT']),
        user=cfg['DB_USER'],
        password=cfg['DB_PASSWORD'],
        database=base_datos,
        connect_timeout=5,
    )
    with contextlib.redirect_stdout(io.StringIO()):
        ex.connect()

    @event.listens_for(ex.engine, 'connect')
    def configurar(dbapi, record):
        # Aplicar el modo de lectura y el aislamiento indicados por esta etapa.
        dbapi.set_session(
            readonly=solo_lectura, isolation_level='REPEATABLE READ', autocommit=False
        )

    @event.listens_for(ex.engine, 'begin')
    def limitar(conn):
        # Evitar que una sentencia o un bloqueo mantenga el proceso esperando sin
        # límite.
        conn.exec_driver_sql("SET LOCAL statement_timeout = '120s'")
        conn.exec_driver_sql("SET LOCAL lock_timeout = '5s'")

    ex.engine.dispose()
    return ex


registrar_estado()
ex_origen = ex_destino = None
try:
    exigir(
        cfg['DB_NAME_SOURCE'] == 'colombia_saludable'
        and cfg['DB_NAME_WAREHOUSE'] == DESTINO_AUTORIZADO,
        'Las bases no coinciden con el origen y destino autorizados.',
    )
    ex_origen = conectar_etl(cfg['DB_NAME_SOURCE'], True)
    # En preparación, también el destino queda protegido contra escritura.
    ex_destino = conectar_etl(cfg['DB_NAME_WAREHOUSE'], not EJECUTAR_CARGA)
    exigir(
        q(ex_origen, 'SELECT current_database() AS db').iloc[0, 0]
        == 'colombia_saludable',
        'Origen incorrecto',
    )
    exigir(
        q(ex_destino, 'SELECT current_database() AS db').iloc[0, 0]
        == DESTINO_AUTORIZADO,
        'Destino incorrecto',
    )
    exigir(
        q(ex_origen, "SHOW transaction_read_only").iloc[0, 0] == 'on',
        'Origen no protegido',
    )
    if EJECUTAR_CARGA:
        # Incluye relaciones, vistas, secuencias y funciones de usuario, no solo public.
        objetos = q(
            ex_destino,
            """SELECT COUNT(*) AS n FROM (
            SELECT c.oid FROM pg_catalog.pg_class c JOIN pg_catalog.pg_namespace n ON n.oid=c.relnamespace
            WHERE n.nspname NOT IN ('pg_catalog','information_schema') AND n.nspname NOT LIKE 'pg_toast%'
              AND n.nspname NOT LIKE 'pg_temp%'
            UNION ALL
            SELECT p.oid FROM pg_catalog.pg_proc p JOIN pg_catalog.pg_namespace n ON n.oid=p.pronamespace
            WHERE n.nspname NOT IN ('pg_catalog','information_schema') AND n.nspname NOT LIKE 'pg_%'
            ) x""",
        ).iloc[0, 0]
        exigir(
            int(objetos) == 0,
            'CARGA_BLOQUEADA_DESTINO_NO_VACIO: no se reemplazó ni agregó información.',
        )
except Exception as error:
    estado_run['estado'] = 'bloqueado_prevalidacion'
    estado_run['motivo'] = (
        'destino_no_vacio'
        if 'CARGA_BLOQUEADA_DESTINO_NO_VACIO' in str(error)
        else 'configuracion_o_conexion'
    )
    registrar_estado()
    for ex in (ex_origen, ex_destino):
        if ex:
            with contextlib.redirect_stdout(io.StringIO()):
                ex.close_connection()
    raise RuntimeError(
        'Prevalidación ETL detenida. Consulte carga_core/intentos/' + RUN_ID + '.json'
    ) from None


# Guardar la huella del Core que se contrastará al finalizar.
CORE_ANTES = huella_core()
print('Modo:', estado_run['modo'], '| origen READ ONLY | destino:', DESTINO_AUTORIZADO)
print('Run:', RUN_ID)


Modo: carga | origen READ ONLY | destino: colombia_saludable_dw_core_v2
Run: 20261030T150852_0357772f


## B1. Extracción y fotografía de entrada

SELECT explícitos mediante el extractor Core, sin transformaciones SQL. Se comprueban las 18 tablas y sus
columnas contra el diccionario aprobado. Se registra una huella global por tabla para detectar cambios
entre la extracción, la prevalidación y el cierre; esto no crea un snapshot global entre tablas.

**Funciones utilizadas en la siguiente celda**

- **Core, directamente o a través de auxiliares:** `CSV_Loader.load_csv`, `DB_Extractor.execute_query`, `DB_Extractor.get_table_names`, `DB_Extractor.get_table_schema`.
- **Fuera del Core y motivo:** Python compara el diccionario aprobado, arma SELECT de columnas explícitas y registra huellas/conteos; pandas solo crea el resumen. La lectura del CSV de metadatos ya usa CSV_Loader.load_csv.

**Funciones propias definidas o llamadas en esta celda**

| Función | Relación con el Core y motivo de conservarla |
| --- | --- |
| `exigir` | comprueba condiciones y emite errores de negocio del taller. |
| `huella_frame` | calcula SHA-256 de una serialización ordenada para comprobar reproducibilidad. |
| `ident` | escapa identificadores SQL del catálogo; no calcula medidas. |
| `q` | delega el SQL a DB_Extractor.execute_query y silencia sus mensajes. |
| `tabla` | construye el nombre SQL public.tabla con identificadores escapados. |

**SQL:** las consultas de esta celda se ejecutan mediante `DB_Extractor.execute_query`, pero sus expresiones SQL pertenecen al cuaderno. Se conserva SQL para perfilado, validación o conciliación; la extracción B1 solo selecciona columnas y la transformación tabular de B2–B5 usa los transformadores Core.

**Llamadas externas directas de esta celda** (incluye los cuerpos de funciones definidos aquí; las dependencias de auxiliares llamados se explican donde se definen; `…` omite argumentos):

| Grupo | Funciones y métodos | Por qué se conservan fuera del Core |
| --- | --- | --- |
| Python estándar y contenedores | `CONTEOS_ORIGEN.items`, `fuentes.items`, `len`, `set`, `set(…).isdisjoint`, `sorted`, `str`, `str.join` | Orquestación, condiciones, iteración, construcción de SQL, parámetros y mensajes. No sustituyen una transformación tabular del Core. |
| Archivos, configuración y reproducibilidad | `contextlib.redirect_stdout`, `io.StringIO` | Rutas, entorno, manifiestos, fechas de ejecución, versiones o hashes específicos de este proceso; el Core no publica estos manifiestos del taller. |
| DataFrames y matrices | `diccionario_aprobado.to_dict`, `pd.DataFrame` | Construcción de evidencia, acceso/iteración de resultados, selección de máximos y presentación de matrices. Los filtros, CSV, conversiones y deduplicación con equivalencia directa se delegan al Core. head del Core devuelve texto; no hay pivot/reindex tabular equivalente en estos transformadores. |
| Dibujo e IPython | `display` | Renderizado, ejes, leyendas, imágenes y visualización del notebook; el Core no ofrece renderizadores Python de gráficos ni display HTML. |

In [12]:
# Extracción de la fotografía de entrada mediante el Core.


# Comparar el catálogo actual con el diccionario aprobado antes de extraer.
diccionario_aprobado = CSV_CORE.load_csv(
    str(EVIDENCIA / 'diccionario_origen.csv'), keep_default_na=False
)
with contextlib.redirect_stdout(io.StringIO()):
    tablas_actuales = sorted(ex_origen.get_table_names())
tablas_aprobadas = sorted(set(diccionario_aprobado['tabla']))
exigir(
    tablas_actuales == tablas_aprobadas,
    'Cambió el catálogo del origen; revisar plan antes de cargar.',
)

# Leer columnas explícitas y conservar en memoria cada tabla de origen.
fuentes = {}
consultas_extraccion = {}
for t in tablas_aprobadas:
    esperadas = [
        r['columna'] for r in diccionario_aprobado.to_dict('records') if r['tabla'] == t
    ]
    with contextlib.redirect_stdout(io.StringIO()):
        reales = ex_origen.get_table_schema(t)['columns']
    exigir(reales == esperadas, 'Cambió el esquema de ' + t)
    consultas_extraccion[t] = (
        'SELECT ' + ', '.join(ident(c) for c in esperadas) + ' FROM ' + tabla(t)
    )
    fuentes[t] = q(ex_origen, consultas_extraccion[t])

# Registrar huellas y conteos para comprobar que no se pierdan ni cambien datos.
HUELLAS_ORIGEN = {t: huella_frame(df) for t, df in fuentes.items()}
CONTEOS_ORIGEN = {t: len(df) for t, df in fuentes.items()}
exigir(
    set(fuentes['cotizante']['cedula']).isdisjoint(
        set(fuentes['beneficiario']['id_beneficiario'])
    ),
    'Existen identificadores de usuario ambiguos entre tipos de persona.',
)
display(
    pd.DataFrame(
        [{'tabla': t, 'filas_extraidas': n} for t, n in CONTEOS_ORIGEN.items()]
    )
)


✅ CSV cargado exitosamente: C:\Users\cabre\Desktop\University\Tesis\python-library-etl-master\python-library-etl-master\colombia_saludable\planes_y_evidencia\diccionario_origen.csv (107 registros, 10 columnas)


,tabla,filas_extraidas
0,beneficiario,3316
1,citas_generales,3013
2,cotizante,1100
3,cotizante_beneficiario,3316
4,drogueria,35
5,empresa,21
6,empresa_cotizante,925
7,formulas_medicas,7484
8,hospitalizaciones,893
9,ips,86


## B2. Mapeos, dimensiones de entidad y calendario

Las categorías de los 67 servicios son un mapeo explícito por código, revisable antes de cargar.
Diagnósticos: normalización NFC, espacios y minúsculas; solo se agrupan unas etiquetas explícitas,
sin inferir CIE-10 ni validar clínicamente el texto. El original se conserva en cada hecho.
Semana Santa significa lunes–domingo de la semana del jueves santo; Navidad, 24–31 de diciembre.
La precedencia es Semana Santa, Navidad y ordinario. No se atribuyen vacaciones.

**Funciones utilizadas en la siguiente celda**

- **Core, directamente o a través de auxiliares:** `BasicsTransformOperations.add_new_column`, `ConvertOperations.convert_column_type`, `DataSelect.select_columns`, `HeaderOperations.rename_columns`, `etl.DateTime`, `etl.schema.missing (alias nulo)`.
- **Fuera del Core y motivo:** Diccionarios Python definen las categorías de servicios/diagnósticos. timedelta y temporada definen convenciones del taller; warnings recoge avisos del calendario. copy e iteración transportan los DataFrames sin reemplazar joins/proyecciones Core.

**Funciones propias definidas o llamadas en esta celda**

| Función | Relación con el Core y motivo de conservarla |
| --- | --- |
| `derivar` | es un envoltorio de BasicsTransformOperations.add_new_column; el callable expresa la regla de negocio. |
| `exigir` | comprueba condiciones y emite errores de negocio del taller. |
| `fecha` | interpreta valores de fecha del origen como auxiliar escalar de las derivaciones. |
| `fecha_id` | define la clave de calendario YYYYMMDD del modelo. |
| `limpiar` | define la regla NFC, espacios, casefold y desconocidos; se suministra como callable a los transformadores Core. |
| `proyectar` | es un envoltorio de DataSelect.select_columns. |
| `renombrar` | es un envoltorio de HeaderOperations.rename_columns. |
| `temporada` | define Semana Santa/Navidad del taller a partir del calendario generado por el Core. |

**Expresiones de fila:** 11 lambda(s) escritas aquí. Su fórmula o condición pertenece al cuaderno; cuando se pasan a un transformador, el Core realiza la operación tabular. No se presentan como funciones propias del Core.

**Llamadas externas directas de esta celda** (incluye los cuerpos de funciones definidos aquí; las dependencias de auxiliares llamados se explican donde se definen; `…` omite argumentos):

| Grupo | Funciones y métodos | Por qué se conservan fuera del Core |
| --- | --- | --- |
| Python estándar y contenedores | `GRUPOS_SERVICIO.items`, `GRUPOS_SERVICIO.values`, `bool`, `calendario.holidays_colombia.items`, `fechas_evento.extend`, `len`, `map`, `max`, `min`, `print`, `set`, `sum` | Orquestación, condiciones, iteración, construcción de SQL, parámetros y mensajes. No sustituyen una transformación tabular del Core. |
| Archivos, configuración y reproducibilidad | `warnings.catch_warnings` | Rutas, entorno, manifiestos, fechas de ejecución, versiones o hashes específicos de este proceso; el Core no publica estos manifiestos del taller. |
| DataFrames y matrices | `calendario.df.copy`, `fuentes[…].copy` | Construcción de evidencia, acceso/iteración de resultados, selección de máximos y presentación de matrices. Los filtros, CSV, conversiones y deduplicación con equivalencia directa se delegan al Core. head del Core devuelve texto; no hay pivot/reindex tabular equivalente en estos transformadores. |
| Expresiones escalares del dominio | `nombre.casefold`, `timedelta` | Normalización Unicode, fechas, duraciones, claves y redondeo Decimal de las reglas explícitas. Se aplican con callables Core cuando corresponden a columnas; no son funciones propias del Core. |

In [13]:
# Mapeos explícitos, dimensiones de entidad y calendario.


# Clasificar servicios por código y exigir que el mapeo cubra todo el catálogo.
GRUPOS_SERVICIO = {
    'consulta_general': ['1234'],
    'consulta_especialista': ['1269'],
    'hospitalizacion': ['1346'],
    'urgencias': ['1367'],
    'cirugia': [
        '1241',
        '1248',
        '1339',
        '1353',
        '1388',
        '1395',
        '1430',
        '1479',
        '1493',
        '1521',
        '1556',
        '1563',
        '1584',
        '1591',
        '1598',
        '1640',
        '1689',
        '1696',
    ],
    'tratamiento': [
        '1297',
        '1304',
        '1311',
        '1451',
        '1549',
        '1605',
        '1626',
        '1633',
        '1647',
        '1668',
        '1682',
    ],
    'examen': [
        '1255',
        '1262',
        '1276',
        '1283',
        '1290',
        '1318',
        '1325',
        '1332',
        '1360',
        '1374',
        '1381',
        '1402',
        '1409',
        '1416',
        '1423',
        '1437',
        '1444',
        '1458',
        '1465',
        '1472',
        '1486',
        '1500',
        '1507',
        '1514',
        '1528',
        '1535',
        '1542',
        '1570',
        '1577',
        '1612',
        '1619',
        '1654',
        '1661',
        '1675',
    ],
}
MAPA_SERVICIO = {
    codigo: grupo for grupo, codigos in GRUPOS_SERVICIO.items() for codigo in codigos
}
exigir(
    len(MAPA_SERVICIO) == sum(map(len, GRUPOS_SERVICIO.values())),
    'Código de servicio mapeado dos veces.',
)
exigir(
    set(MAPA_SERVICIO) == set(fuentes['servicios_pos']['id_servicio_pos']),
    'Mapeo de servicios incompleto o desactualizado.',
)

# Agrupar únicamente etiquetas previstas; esta clasificación no valida diagnósticos.
MAPA_DIAGNOSTICO = {
    'vih': ('vih_sida', 'diagnostico_observado'),
    'sida': ('vih_sida', 'diagnostico_observado'),
    'deficiencia renal': ('deficiencia_renal', 'diagnostico_observado'),
    'deficiencia renal severa': ('deficiencia_renal', 'diagnostico_observado'),
    'control deficiencia renal': ('deficiencia_renal', 'seguimiento'),
    'diabetes': ('diabetes', 'diagnostico_observado'),
    'control diabetes': ('diabetes', 'seguimiento'),
    'cancer glótico': ('cancer', 'diagnostico_observado'),
    'cancer laringe': ('cancer', 'diagnostico_observado'),
    'sin novedad': (None, 'control_o_evolucion'),
    'evolución satisfactoria': (None, 'control_o_evolucion'),
    'evolución satisfctoria': (None, 'control_o_evolucion'),
    'hemodiálisis sin novedad': (None, 'seguimiento_procedimiento'),
    'diálisis sin novedad': (None, 'seguimiento_procedimiento'),
}


# Preparar la dimensión de empresas conservando los textos originales.
dim_empresa = renombrar(
    proyectar(
        fuentes['empresa'],
        'nit',
        'nombre',
        'actividad_economica',
        'ciudad',
        'total_empleados',
    ),
    {
        'nit': 'empresa_id',
        'ciudad': 'ciudad_original',
        'actividad_economica': 'actividad_economica_original',
    },
)
for original, nueva in [
    ('ciudad_original', 'ciudad'),
    ('actividad_economica_original', 'actividad_economica'),
]:
    dim_empresa = derivar(dim_empresa, nueva, lambda r, c=original: limpiar(r[c]))


# Normalizar el catálogo de IPS; su geografía no representa residencia del paciente.
dim_ips = renombrar(
    proyectar(
        fuentes['ips'],
        'id_ips',
        'nombre',
        'tipo_ips',
        'nivel',
        'municipio',
        'departamento',
    ),
    {
        'id_ips': 'ips_id',
        **{
            c: c + '_original'
            for c in ['tipo_ips', 'nivel', 'municipio', 'departamento']
        },
    },
)
for c in ['tipo_ips', 'nivel', 'municipio', 'departamento']:
    dim_ips = derivar(dim_ips, c, lambda r, c=c: limpiar(r[c + '_original']))


# Conservar especialidades originales y señalar la falta de historia de IPS.
dim_medico = renombrar(
    proyectar(fuentes['medico'], 'cedula', 'especialidad', 'subespecialidad', 'id_ips'),
    {
        'cedula': 'medico_id',
        'id_ips': 'ips_id',
        'especialidad': 'especialidad_original',
        'subespecialidad': 'subespecialidad_original',
    },
)
for c in ['especialidad', 'subespecialidad']:
    dim_medico = derivar(dim_medico, c, lambda r, c=c: limpiar(r[c + '_original']))
dim_medico = derivar(dim_medico, 'ips_sin_vigencia_historica', lambda r: True)

# Preparar el catálogo de droguerías y normalizar su ciudad.
dim_drogueria = renombrar(
    proyectar(fuentes['drogueria'], 'id_drogueria', 'nombre', 'ciudad'),
    {'id_drogueria': 'drogueria_id', 'ciudad': 'ciudad_original'},
)
dim_drogueria = derivar(
    dim_drogueria, 'ciudad', lambda r: limpiar(r['ciudad_original'])
)


# Conservar la tarifa de catálogo y registrar sus límites de moneda/vigencia.
dim_servicio = renombrar(
    fuentes['servicios_pos'].copy(),
    {
        'id_servicio_pos': 'servicio_id',
        'descripcion': 'descripcion_original',
        'costo': 'tarifa_catalogo',
    },
)
dim_servicio = derivar(
    dim_servicio, 'descripcion', lambda r: limpiar(r['descripcion_original'])
)
dim_servicio = derivar(
    dim_servicio, 'tipo_servicio', lambda r: MAPA_SERVICIO[r['servicio_id']]
)
dim_servicio = derivar(dim_servicio, 'moneda_informada', lambda r: False)
dim_servicio = derivar(dim_servicio, 'precio_sin_vigencia_historica', lambda r: True)


# Cubrir con el calendario Core los años presentes en los hechos de origen.
fechas_evento = []
for t, c in [
    ('citas_generales', 'fecha_solicitud'),
    ('citas_generales', 'fecha_atencion'),
    ('urgencias', 'fecha_solicitud'),
    ('urgencias', 'fecha_atencion'),
    ('hospitalizaciones', 'fecha_solicitud'),
    ('hospitalizaciones', 'fecha_atencion'),
    ('remisiones', 'fecha_remision'),
    ('remisiones', 'fecha_atencion'),
    ('pagos', 'fecha_pago'),
    ('formulas_medicas', 'fecha'),
    ('retiros', 'fecha_retiro'),
]:
    fechas_evento.extend(fecha(v) for v in fuentes[t][c] if not nulo(v))
exigir(bool(fechas_evento), 'No hay fechas para el calendario')
with warnings.catch_warnings(record=True) as avisos_calendario:
    calendario = CalendarioCore(min(fechas_evento).year, max(fechas_evento).year)
    dim_fecha = calendario.df.copy()
dim_fecha = renombrar(
    dim_fecha,
    {
        'año': 'anio',
        'día': 'dia',
        'semana_del_año': 'semana_anio',
        'día_de_la_semana': 'dia_semana',
        'nombre_día': 'nombre_dia',
    },
)
dim_fecha = derivar(dim_fecha, 'fecha_id', lambda r: fecha_id(r['fecha']))
dim_fecha = S.select_columns(dim_fecha, 'id', complement=True, show=0)
dim_fecha = C.convert_column_type(dim_fecha, ['fecha'], 'datetime', show=0)
dim_fecha = C.convert_column_type(
    dim_fecha, ['es_fin_de_semana', 'es_festivo'], 'bool', show=0
)

# Derivar temporadas convencionales a partir de los festivos del calendario.
semanas_santas = {
    d.year: (d - timedelta(days=3), d + timedelta(days=3))
    for d, nombre in calendario.holidays_colombia.items()
    if 'jueves santo' in nombre.casefold()
}


def temporada(f):
    # Aplicar la convención documentada: Semana Santa, Navidad y luego ordinario.
    d = fecha(f)
    a, b = semanas_santas[d.year]
    return (
        'semana_santa'
        if a <= d <= b
        else ('navidad' if d.month == 12 and d.day >= 24 else 'ordinario')
    )


dim_fecha = derivar(
    dim_fecha, 'temporada_convencional', lambda r: temporada(r['fecha'])
)
print(
    'Dimensiones de entidad y calendario preparadas; mapeo de servicios:',
    len(MAPA_SERVICIO),
)


Dimensiones de entidad y calendario preparadas; mapeo de servicios: 67


## B3. Familia y perfil de pacientes

Se conserva el tipo de registro con prefijos C/B. La relación del beneficiario con el titular aporta atributos
familiares explícitamente nombrados, sin atribuirle salario, escolaridad o fecha de afiliación propios.
No se cargan nombres ni direcciones de personas. Un join no puede multiplicar filas.

**Funciones utilizadas en la siguiente celda**

- **Core, directamente o a través de auxiliares:** `BasicsTransformOperations.add_new_column`, `ConvertOperations.convert_column_type`, `DataSelect.filter_in_list`, `DataSelect.filter_is_null`, `DataSelect.select_columns`, `HeaderOperations.rename_columns`, `TransformOperations.drop_duplicates`, `TransformOperations.group_by_count`, `TransformOperations.left_join`, `TransformOperations.union_all`, `etl.schema.missing (alias nulo)`.
- **Fuera del Core y motivo:** Las lambdas definen claves C:/B:/F:, atributos heredados y desconocidos. El Core ejecuta cruces, agrupación, unión y conversiones; el alias boolean actual conserva NULL sin pd.BooleanDtype externo.

**Funciones propias definidas o llamadas en esta celda**

| Función | Relación con el Core y motivo de conservarla |
| --- | --- |
| `clave` | define el tratamiento de claves nulas y espacios como expresión de fila. |
| `derivar` | es un envoltorio de BasicsTransformOperations.add_new_column; el callable expresa la regla de negocio. |
| `limpiar` | define la regla NFC, espacios, casefold y desconocidos; se suministra como callable a los transformadores Core. |
| `proyectar` | es un envoltorio de DataSelect.select_columns. |
| `renombrar` | es un envoltorio de HeaderOperations.rename_columns. |
| `unico` | delega selección, nulos y deduplicación al Core y añade los mensajes/valores inválidos del contrato. |
| `unir_izquierda` | delega a TransformOperations.left_join y añade comprobaciones de cardinalidad. |

**Expresiones de fila:** 24 lambda(s) escritas aquí. Su fórmula o condición pertenece al cuaderno; cuando se pasan a un transformador, el Core realiza la operación tabular. No se presentan como funciones propias del Core.

**Llamadas externas directas de esta celda** (incluye los cuerpos de funciones definidos aquí; las dependencias de auxiliares llamados se explican donde se definen; `…` omite argumentos):

| Grupo | Funciones y métodos | Por qué se conservan fuera del Core |
| --- | --- | --- |
| Python estándar y contenedores | `dict.items`, `int`, `len`, `print` | Orquestación, condiciones, iteración, construcción de SQL, parámetros y mensajes. No sustituyen una transformación tabular del Core. |
| DataFrames y matrices | `fuentes[…].copy` | Construcción de evidencia, acceso/iteración de resultados, selección de máximos y presentación de matrices. Los filtros, CSV, conversiones y deduplicación con equivalencia directa se delegan al Core. head del Core devuelve texto; no hay pivot/reindex tabular equivalente en estos transformadores. |

In [14]:
# Construcción de familias, cotizantes y beneficiarios.


# Contar los beneficiarios por titular sin duplicar las familias.
familias_por_cotizante = T.group_by_count(
    fuentes['cotizante_beneficiario'], by='cotizante', show=0
)
dim_familia = renombrar(
    proyectar(fuentes['cotizante'], 'cedula'), {'cedula': 'cotizante_origen'}
)
dim_familia = unir_izquierda(
    dim_familia, familias_por_cotizante, ('cotizante_origen', 'cotizante'), 'familias'
)
dim_familia = derivar(dim_familia, 'familia_id', lambda r: 'F:' + r['cotizante_origen'])
dim_familia = derivar(
    dim_familia, 'paciente_titular_id', lambda r: 'C:' + r['cotizante_origen']
)
dim_familia = derivar(
    dim_familia,
    'numero_beneficiarios',
    lambda r: 0 if nulo(r['conteo']) else int(r['conteo']),
)
dim_familia = derivar(
    dim_familia, 'numero_miembros', lambda r: 1 + r['numero_beneficiarios']
)
dim_familia = proyectar(
    dim_familia,
    'familia_id',
    'paciente_titular_id',
    'numero_beneficiarios',
    'numero_miembros',
)


# Preparar la relación empresarial y los atributos propios del cotizante.
laboral = renombrar(
    fuentes['empresa_cotizante'].copy(),
    {'empresa': 'empresa_id', 'cotizante': 'usuario_origen'},
)
cot = renombrar(
    proyectar(
        fuentes['cotizante'],
        'cedula',
        'sexo',
        'fecha_nacimiento',
        'estado_civil',
        'tipo_discapacidad',
        'tipo_cotizante',
        'nivel_escolaridad',
        'estracto',
        'proviene_otra_eps',
        'salario_base',
        'fecha_afiliacion',
        'id_ips',
    ),
    {
        'cedula': 'usuario_origen',
        'sexo': 'sexo_original',
        'estado_civil': 'estado_civil_original',
        'tipo_discapacidad': 'tipo_discapacidad_original',
        'estracto': 'estrato_original',
        'tipo_cotizante': 'tipo_cotizante_original',
        'nivel_escolaridad': 'escolaridad_original',
        'proviene_otra_eps': 'proviene_otra_eps_original',
        'salario_base': 'salario_base_propio',
        'fecha_afiliacion': 'fecha_afiliacion_propia',
        'id_ips': 'ips_afiliacion_id',
    },
)
cot = unir_izquierda(cot, laboral, 'usuario_origen', 'empresa actual por cotizante')
for c, fn in {
    'paciente_id': lambda r: 'C:' + r['usuario_origen'],
    'tipo_registro': lambda r: 'cotizante',
    'familia_id': lambda r: 'F:' + r['usuario_origen'],
    'paciente_titular_id': lambda r: r['paciente_id'],
    'tipo_documento_original': lambda r: None,
    'parentesco_original': lambda r: None,
    'estrato_propio': lambda r: clave(r['estrato_original']),
    'estrato_familiar': lambda r: clave(r['estrato_original']),
    'tipo_cotizante_propio': lambda r: limpiar(r['tipo_cotizante_original']),
    'tipo_cotizante_titular': lambda r: limpiar(r['tipo_cotizante_original']),
    'escolaridad_propia': lambda r: limpiar(r['escolaridad_original']),
    'proviene_otra_eps_propio': lambda r: (
        None
        if nulo(r['proviene_otra_eps_original'])
        else r['proviene_otra_eps_original'] == '1'
    ),
    'fecha_afiliacion_titular': lambda r: r['fecha_afiliacion_propia'],
    'perfil_familiar_heredado': lambda r: False,
}.items():
    cot = derivar(cot, c, fn)


# Preparar beneficiarios sin atribuirles salario o afiliación propios inexistentes.
ben = renombrar(
    proyectar(
        fuentes['beneficiario'],
        'id_beneficiario',
        'tipo_identificacion',
        'parentesco',
        'sexo',
        'fecha_nacimiento',
        'estado_civil',
        'tipo_discapacidad',
    ),
    {
        'id_beneficiario': 'usuario_origen',
        'tipo_identificacion': 'tipo_documento_original',
        'parentesco': 'parentesco_original',
        'sexo': 'sexo_original',
        'estado_civil': 'estado_civil_original',
        'tipo_discapacidad': 'tipo_discapacidad_original',
    },
)
puente_familia = renombrar(
    fuentes['cotizante_beneficiario'].copy(),
    {'cotizante': 'titular_origen', 'beneficiario': 'usuario_origen'},
)
ben = unir_izquierda(ben, puente_familia, 'usuario_origen', 'titular por beneficiario')

# Heredar solo atributos familiares, identificando su procedencia del titular.
perfil_titular = renombrar(
    proyectar(
        cot,
        'usuario_origen',
        'familia_id',
        'paciente_titular_id',
        'empresa_id',
        'ips_afiliacion_id',
        'estrato_familiar',
        'tipo_cotizante_titular',
        'fecha_afiliacion_titular',
    ),
    {'usuario_origen': 'titular_origen'},
)
ben = unir_izquierda(ben, perfil_titular, 'titular_origen', 'perfil del titular')
for c, fn in {
    'paciente_id': lambda r: 'B:' + r['usuario_origen'],
    'tipo_registro': lambda r: 'beneficiario',
    'perfil_familiar_heredado': lambda r: True,
}.items():
    ben = derivar(ben, c, fn)
for c in [
    'estrato_original',
    'estrato_propio',
    'tipo_cotizante_original',
    'tipo_cotizante_propio',
    'escolaridad_original',
    'escolaridad_propia',
    'proviene_otra_eps_original',
    'proviene_otra_eps_propio',
    'salario_base_propio',
    'fecha_afiliacion_propia',
]:
    ben = derivar(ben, c, lambda r: None)

# Unificar ambos tipos de persona con el mismo conjunto y orden de columnas.
cols_paciente = [
    'paciente_id',
    'usuario_origen',
    'tipo_registro',
    'tipo_documento_original',
    'parentesco_original',
    'sexo_original',
    'fecha_nacimiento',
    'estado_civil_original',
    'tipo_discapacidad_original',
    'familia_id',
    'paciente_titular_id',
    'empresa_id',
    'ips_afiliacion_id',
    'estrato_original',
    'estrato_propio',
    'estrato_familiar',
    'tipo_cotizante_original',
    'tipo_cotizante_propio',
    'tipo_cotizante_titular',
    'escolaridad_original',
    'escolaridad_propia',
    'proviene_otra_eps_original',
    'proviene_otra_eps_propio',
    'salario_base_propio',
    'fecha_afiliacion_propia',
    'fecha_afiliacion_titular',
    'perfil_familiar_heredado',
]
dim_paciente = T.union_all(
    [proyectar(cot, *cols_paciente), proyectar(ben, *cols_paciente)], show=0
)
for c in ['sexo', 'estado_civil', 'tipo_discapacidad']:
    dim_paciente = derivar(dim_paciente, c, lambda r, c=c: limpiar(r[c + '_original']))
dim_paciente = derivar(dim_paciente, 'atribucion_familiar_sin_historia', lambda r: True)
dim_paciente = C.convert_column_type(
    dim_paciente, ['salario_base_propio'], 'Int64', show=0
)
# El Core actual convierte 'boolean' a un tipo nullable y conserva los nulos.
dim_paciente = C.convert_column_type(
    dim_paciente, ['proviene_otra_eps_propio'], 'boolean', show=0
)
dim_paciente = C.convert_column_type(
    dim_paciente,
    ['fecha_nacimiento', 'fecha_afiliacion_propia', 'fecha_afiliacion_titular'],
    'datetime',
    show=0,
)
unico(dim_paciente, ['usuario_origen'], 'resolución de usuario entre ambos tipos')
print('Pacientes:', len(dim_paciente), '| familias:', len(dim_familia))


Pacientes: 4416 | familias: 1100


## B4. Recetas, catálogo de códigos y diagnósticos

La expansión de recetas conserva su cabecera. Cada par fórmula–medicamento significa presencia, no dosis
ni unidades entregadas. Los nombres y precios de medicamento permanecen nulos porque no existe catálogo.

**Funciones utilizadas en la siguiente celda**

- **Core, directamente o a través de auxiliares:** `BasicsTransformOperations.add_new_column`, `BasicsTransformOperations.transform_column`, `ConvertOperations.convert_column_type`, `DataExpresion.split_column_into_rows`, `DataSelect.select_columns`, `TransformOperations.drop_duplicates`, `TransformOperations.union_all`, `etl.schema.missing (alias nulo)`.
- **Fuera del Core y motivo:** hashlib y el diccionario MAPA_DIAGNOSTICO definen identificadores y clasificación textual específica. Las comprobaciones Python evitan inventar catálogos, nombres o precios; la expansión y deduplicación son Core.

**Funciones propias definidas o llamadas en esta celda**

| Función | Relación con el Core y motivo de conservarla |
| --- | --- |
| `derivar` | es un envoltorio de BasicsTransformOperations.add_new_column; el callable expresa la regla de negocio. |
| `dx_id` | genera una clave diagnóstica SHA-256 determinista, específica del modelo. |
| `exigir` | comprueba condiciones y emite errores de negocio del taller. |
| `limpiar` | define la regla NFC, espacios, casefold y desconocidos; se suministra como callable a los transformadores Core. |
| `proyectar` | es un envoltorio de DataSelect.select_columns. |

**Expresiones de fila:** 10 lambda(s) escritas aquí. Su fórmula o condición pertenece al cuaderno; cuando se pasan a un transformador, el Core realiza la operación tabular. No se presentan como funciones propias del Core.

**Llamadas externas directas de esta celda** (incluye los cuerpos de funciones definidos aquí; las dependencias de auxiliares llamados se explican donde se definen; `…` omite argumentos):

| Grupo | Funciones y métodos | Por qué se conservan fuera del Core |
| --- | --- | --- |
| Python estándar y contenedores | `MAPA_DIAGNOSTICO.get`, `all`, `dict.items`, `len`, `limpiar(…).encode`, `print`, `set`, `set(…).isdisjoint` | Orquestación, condiciones, iteración, construcción de SQL, parámetros y mensajes. No sustituyen una transformación tabular del Core. |
| Archivos, configuración y reproducibilidad | `hashlib.sha256`, `hashlib.sha256(…).hexdigest` | Rutas, entorno, manifiestos, fechas de ejecución, versiones o hashes específicos de este proceso; el Core no publica estos manifiestos del taller. |

In [15]:
# Expansión de recetas y construcción del catálogo diagnóstico.


# Separar medicamentos por receta y limpiar cada código con el Core.
detalle_formula = proyectar(
    fuentes['formulas_medicas'], 'codigo_formula', 'medicamentos_recetados'
)
detalle_formula = X.split_column_into_rows(
    detalle_formula,
    'medicamentos_recetados',
    ';',
    show=0,
    new_column_name='medicamento_id',
    drop_original=True,
)
detalle_formula = B.transform_column(detalle_formula, 'medicamento_id', clave, show=0)
exigir(
    all(v for v in detalle_formula['medicamento_id']),
    'Hay tokens vacíos de medicamento; revisar origen.',
)

# Deduplicar el par receta-código: representa presencia, no dosis.
puente_formula_medicamento = T.drop_duplicates(
    detalle_formula, subset=['codigo_formula', 'medicamento_id'], show=0
)
puente_formula_medicamento = derivar(
    puente_formula_medicamento, 'presencia_en_receta', lambda r: 1
)

# Conservar códigos observados sin inventar nombres o precios.
dim_medicamento = T.drop_duplicates(
    proyectar(detalle_formula, 'medicamento_id'), subset=['medicamento_id'], show=0
)
for c, fn in {
    'nombre': lambda r: None,
    'precio': lambda r: None,
    'sin_catalogo': lambda r: True,
    'cantidad_informada': lambda r: False,
}.items():
    dim_medicamento = derivar(dim_medicamento, c, fn)
exigir(
    set(dim_medicamento['medicamento_id']).isdisjoint(set(dim_servicio['servicio_id'])),
    'Cambió la cobertura del catálogo de medicamentos; revisar el plan.',
)
# Dtypes nullable: columnas enteramente nulas no deben convertirse a texto "None".
dim_medicamento = C.convert_column_type(dim_medicamento, ['nombre'], 'string', show=0)
dim_medicamento = C.convert_column_type(dim_medicamento, ['precio'], 'Int64', show=0)


# Reunir los textos de las cuatro fuentes de atención para normalizarlos.
texto_diag = T.union_all(
    [
        proyectar(fuentes[t], 'diagnostico')
        for t in ['citas_generales', 'urgencias', 'hospitalizaciones', 'remisiones']
    ],
    show=0,
)
texto_diag = derivar(
    texto_diag, 'diagnostico_normalizado', lambda r: limpiar(r['diagnostico'])
)
exigir(
    all(not nulo(v) for v in texto_diag['diagnostico_normalizado']),
    'Diagnóstico vacío sin regla de desconocido.',
)


def dx_id(v):
    # Dar la misma clave a los diagnósticos que comparten el texto normalizado.
    return 'DX:' + hashlib.sha256(limpiar(v).encode('utf8')).hexdigest()


texto_diag = derivar(texto_diag, 'diagnostico_id', lambda r: dx_id(r['diagnostico']))

# Aplicar el mapeo explícito e identificar textos no clasificados.
dim_diagnostico = T.drop_duplicates(
    proyectar(texto_diag, 'diagnostico_id', 'diagnostico_normalizado'),
    subset=['diagnostico_id'],
    show=0,
)
dim_diagnostico = derivar(
    dim_diagnostico,
    'grupo_textual',
    lambda r: MAPA_DIAGNOSTICO.get(
        r['diagnostico_normalizado'], (None, 'no_clasificado')
    )[0],
)
dim_diagnostico = derivar(
    dim_diagnostico,
    'clase_texto',
    lambda r: MAPA_DIAGNOSTICO.get(
        r['diagnostico_normalizado'], (None, 'no_clasificado')
    )[1],
)
dim_diagnostico = derivar(dim_diagnostico, 'validado_clinicamente', lambda r: False)
mapeo_diagnosticos = T.drop_duplicates(
    proyectar(texto_diag, 'diagnostico', 'diagnostico_normalizado', 'diagnostico_id'),
    show=0,
)
print(
    'Fórmulas:',
    len(fuentes['formulas_medicas']),
    '| pares únicos:',
    len(puente_formula_medicamento),
    '| códigos:',
    len(dim_medicamento),
    '| diagnósticos normalizados:',
    len(dim_diagnostico),
)


Fórmulas: 7484 | pares únicos: 15770 | códigos: 166 | diagnósticos normalizados: 87


## B5. Hechos detallados y medidas

Se distinguen IPS asociada actualmente al médico e IPS de afiliación. El origen no certifica la IPS real
del evento. La espera utiliza timestamps locales de solicitud/remisión y atención en America/Bogota;
las diferencias negativas se conservan crudas y quedan nulas en la medida válida.
La edad al evento usa atención; también se guarda edad a solicitud para preguntas de demanda.
Las tarifas son estimaciones de catálogo únicamente en remisiones enlazadas, sin imputación a otros eventos.

**Funciones utilizadas en la siguiente celda**

- **Core, directamente o a través de auxiliares:** `BasicsTransformOperations.add_new_column`, `BasicsTransformOperations.transform_column`, `ConvertOperations.convert_column_type`, `DataSelect.filter_in_list`, `DataSelect.filter_is_null`, `DataSelect.select_columns`, `HeaderOperations.rename_columns`, `TransformOperations.drop_duplicates`, `TransformOperations.left_join`, `TransformOperations.union_all`, `etl.schema.missing (alias nulo)`.
- **Fuera del Core y motivo:** Las expresiones de fila usan fechas, Decimal, hashlib y JSON para reglas de edad, espera, importes y claves. Python configura los esquemas de cada tipo de hecho; las operaciones tabulares se delegan al Core.

**Funciones propias definidas o llamadas en esta celda**

| Función | Relación con el Core y motivo de conservarla |
| --- | --- |
| `derivar` | es un envoltorio de BasicsTransformOperations.add_new_column; el callable expresa la regla de negocio. |
| `dx_id` | genera una clave diagnóstica SHA-256 determinista, específica del modelo. |
| `edad` | define años cumplidos entre nacimiento y evento como expresión de fila del ETL. |
| `enriquecer_medico` | delega el cruce de IPS al Core mediante unir_izquierda. |
| `enriquecer_usuario` | delega el cruce al Core mediante unir_izquierda y exige correspondencia de pacientes. |
| `exigir` | comprueba condiciones y emite errores de negocio del taller. |
| `fecha` | interpreta valores de fecha del origen como auxiliar escalar de las derivaciones. |
| `fecha_id` | define la clave de calendario YYYYMMDD del modelo. |
| `grupo_edad` | define los intervalos y la categoría desconocido del taller. |
| `limpiar` | define la regla NFC, espacios, casefold y desconocidos; se suministra como callable a los transformadores Core. |
| `minutos` | define diferencias y redondeo Decimal de seis posiciones para las esperas. |
| `proyectar` | es un envoltorio de DataSelect.select_columns. |
| `renombrar` | es un envoltorio de HeaderOperations.rename_columns. |
| `sello` | combina fecha y hora locales; el calendario Core no proporciona este auxiliar escalar. |
| `unir_izquierda` | delega a TransformOperations.left_join y añade comprobaciones de cardinalidad. |

**Expresiones de fila:** 32 lambda(s) escritas aquí. Su fórmula o condición pertenece al cuaderno; cuando se pasan a un transformador, el Core realiza la operación tabular. No se presentan como funciones propias del Core.

**Llamadas externas directas de esta celda** (incluye los cuerpos de funciones definidos aquí; las dependencias de auxiliares llamados se explican donde se definen; `…` omite argumentos):

| Grupo | Funciones y métodos | Por qué se conservan fuera del Core |
| --- | --- | --- |
| Python estándar y contenedores | `all`, `dict.items`, `eventos.append`, `print` | Orquestación, condiciones, iteración, construcción de SQL, parámetros y mensajes. No sustituyen una transformación tabular del Core. |
| Archivos, configuración y reproducibilidad | `hashlib.sha256`, `hashlib.sha256(…).hexdigest`, `json.dumps`, `json.dumps(…).encode` | Rutas, entorno, manifiestos, fechas de ejecución, versiones o hashes específicos de este proceso; el Core no publica estos manifiestos del taller. |
| DataFrames y matrices | `fuentes[…].copy` | Construcción de evidencia, acceso/iteración de resultados, selección de máximos y presentación de matrices. Los filtros, CSV, conversiones y deduplicación con equivalencia directa se delegan al Core. head del Core devuelve texto; no hay pivot/reindex tabular equivalente en estos transformadores. |

In [16]:
# Construcción de hechos detallados y medidas derivadas.


# Preparar las columnas de paciente e IPS utilizadas al enriquecer los hechos.
perfil_hecho = proyectar(
    dim_paciente,
    'usuario_origen',
    'paciente_id',
    'familia_id',
    'empresa_id',
    'ips_afiliacion_id',
    'fecha_nacimiento',
    'fecha_afiliacion_propia',
)
ips_medico = renombrar(
    proyectar(dim_medico, 'medico_id', 'ips_id'), {'ips_id': 'ips_medico_asociado_id'}
)


def enriquecer_usuario(df):
    # Añadir el perfil del paciente y rechazar hechos sin correspondencia.
    df = unir_izquierda(df, perfil_hecho, 'usuario_origen', 'paciente del hecho')
    exigir(
        all(not nulo(v) for v in df['paciente_id']),
        'Hay usuarios de hechos sin correspondencia.',
    )
    return df


def enriquecer_medico(df):
    # Añadir la IPS actualmente asociada al médico, sin afirmar vigencia histórica.
    return unir_izquierda(df, ips_medico, 'medico_id', 'IPS del médico')


# Homologar las cuatro fuentes de atención antes de unirlas con el Core.
eventos = []
for t, codigo, tipo in [
    ('citas_generales', 'codigo_cita', 'cita_general'),
    ('urgencias', 'codigo_urgencia', 'urgencia'),
    ('hospitalizaciones', 'codigo_hospitalizacion', 'hospitalizacion'),
    ('remisiones', 'codigo_remision', 'remision'),
]:
    cols = [
        codigo,
        'id_usuario',
        'id_medico',
        'fecha_atencion',
        'hora_atencion',
        'diagnostico',
    ]
    cols += (
        ['fecha_remision', 'hora_remision', 'servicio_pos', 'id_medico_remite']
        if t == 'remisiones'
        else ['fecha_solicitud', 'hora_solicitud']
    )
    if t == 'hospitalizaciones':
        cols += ['duracion_hospitalizacion']
    e = renombrar(
        proyectar(fuentes[t], *cols),
        {
            codigo: 'codigo_origen',
            'id_usuario': 'usuario_origen',
            'id_medico': 'medico_id',
            'diagnostico': 'diagnostico_original',
            'fecha_remision': 'fecha_solicitud',
            'hora_remision': 'hora_solicitud',
            'servicio_pos': 'servicio_id',
            'id_medico_remite': 'medico_remitente_id',
        },
    )
    for c in ['servicio_id', 'medico_remitente_id', 'duracion_hospitalizacion']:
        if c not in e.columns:
            e = derivar(e, c, lambda r: None)
    e = derivar(e, 'tabla_origen', lambda r, t=t: t)
    e = derivar(e, 'tipo_atencion', lambda r, tipo=tipo: tipo)
    e = derivar(e, 'atencion_id', lambda r, t=t: t + ':' + r['codigo_origen'])
    eventos.append(e)

# Unir los eventos y enriquecerlos sin multiplicar filas.
fact_atencion = T.union_all(eventos, show=0)
fact_atencion = enriquecer_medico(enriquecer_usuario(fact_atencion))

# Añadir tarifas solo cuando existe un servicio de catálogo enlazado.
tarifas = proyectar(dim_servicio, 'servicio_id', 'tarifa_catalogo', 'tipo_servicio')
tarifas = B.transform_column(tarifas, 'tarifa_catalogo', dinero, show=0)
fact_atencion = unir_izquierda(fact_atencion, tarifas, 'servicio_id', 'servicio POS')
fact_atencion = renombrar(
    fact_atencion, {'tarifa_catalogo': 'tarifa_servicio_catalogo'}
)
fact_atencion = B.transform_column(
    fact_atencion, 'tarifa_servicio_catalogo', dinero, show=0
)
for c, fn in {
    'diagnostico_id': lambda r: dx_id(r['diagnostico_original']),
    'fecha_solicitud_id': lambda r: fecha_id(r['fecha_solicitud']),
    'fecha_atencion_id': lambda r: fecha_id(r['fecha_atencion']),
    'timestamp_solicitud_local': lambda r: sello(
        r['fecha_solicitud'], r['hora_solicitud']
    ),
    'timestamp_atencion_local': lambda r: sello(
        r['fecha_atencion'], r['hora_atencion']
    ),
    'espera_minutos_cruda': lambda r: minutos(
        r['timestamp_solicitud_local'], r['timestamp_atencion_local']
    ),
    'espera_incompleta': lambda r: nulo(r['espera_minutos_cruda']),
    'espera_negativa': lambda r: (
        False if nulo(r['espera_minutos_cruda']) else r['espera_minutos_cruda'] < 0
    ),
    'espera_valida_min': lambda r: (
        None
        if r['espera_incompleta'] or r['espera_negativa']
        else r['espera_minutos_cruda']
    ),
    'edad_solicitud': lambda r: edad(r['fecha_nacimiento'], r['fecha_solicitud']),
    'edad_al_evento': lambda r: edad(r['fecha_nacimiento'], r['fecha_atencion']),
    'grupo_etario_solicitud': lambda r: grupo_edad(r['edad_solicitud']),
    'grupo_etario_atencion': lambda r: grupo_edad(r['edad_al_evento']),
    'tarifa_informada': lambda r: not nulo(r['tarifa_servicio_catalogo']),
    'duracion_unidad_informada': lambda r: False,
    'ips_medico_sin_vigencia_historica': lambda r: True,
}.items():
    fact_atencion = derivar(fact_atencion, c, fn)
fact_atencion = S.select_columns(
    fact_atencion,
    'fecha_nacimiento',
    'fecha_afiliacion_propia',
    complement=True,
    show=0,
)
fact_atencion = C.convert_column_type(
    fact_atencion,
    ['duracion_hospitalizacion', 'edad_solicitud', 'edad_al_evento'],
    'Int64',
    show=0,
)


# Conservar los aportes recibidos con precisión Decimal.
fact_pago = renombrar(
    fuentes['pagos'].copy(),
    {'id_usuario': 'usuario_origen', 'valor_pagado': 'valor_aporte'},
)
fact_pago = enriquecer_usuario(fact_pago)
fact_pago = B.transform_column(fact_pago, 'valor_aporte', dinero, show=0)
fact_pago = derivar(fact_pago, 'fecha_pago_id', lambda r: fecha_id(r['fecha_pago']))
fact_pago = S.select_columns(
    fact_pago, 'fecha_nacimiento', 'fecha_afiliacion_propia', complement=True, show=0
)


# Conservar la cabecera de cada receta y su edad al evento.
fact_formula = renombrar(
    fuentes['formulas_medicas'].copy(),
    {
        'id_usuario': 'usuario_origen',
        'id_medico': 'medico_id',
        'fecha': 'fecha_formula',
        'medicamentos_recetados': 'medicamentos_recetados_original',
    },
)
fact_formula = enriquecer_medico(enriquecer_usuario(fact_formula))
fact_formula = derivar(
    fact_formula, 'fecha_formula_id', lambda r: fecha_id(r['fecha_formula'])
)
fact_formula = derivar(
    fact_formula,
    'edad_al_evento',
    lambda r: edad(r['fecha_nacimiento'], r['fecha_formula']),
)
fact_formula = derivar(
    fact_formula, 'grupo_etario', lambda r: grupo_edad(r['edad_al_evento'])
)
fact_formula = S.select_columns(
    fact_formula, 'fecha_nacimiento', 'fecha_afiliacion_propia', complement=True, show=0
)


# Separar destino informado de desconocido, sin inferir mora.
fact_retiro = renombrar(
    fuentes['retiros'].copy(),
    {'id_usuario': 'usuario_origen', 'cambio_a_eps': 'eps_destino_original'},
)
fact_retiro = enriquecer_usuario(fact_retiro)
fact_retiro = derivar(
    fact_retiro, 'eps_destino', lambda r: limpiar(r['eps_destino_original'])
)
fact_retiro = derivar(
    fact_retiro, 'destino_informado', lambda r: not nulo(r['eps_destino'])
)
fact_retiro = derivar(
    fact_retiro, 'fecha_retiro_id', lambda r: fecha_id(r['fecha_retiro'])
)
fact_retiro = derivar(
    fact_retiro,
    'edad_al_retiro',
    lambda r: edad(r['fecha_nacimiento'], r['fecha_retiro']),
)
fact_retiro = derivar(
    fact_retiro,
    'dias_afiliacion_al_retiro',
    lambda r: (
        None
        if nulo(r['fecha_afiliacion_propia'])
        else (fecha(r['fecha_retiro']) - fecha(r['fecha_afiliacion_propia'])).days
    ),
)
fact_retiro = S.select_columns(fact_retiro, 'fecha_nacimiento', complement=True, show=0)


# Mantener antecedentes familiares separados de diagnósticos del paciente.
fact_antecedente_familiar = renombrar(
    fuentes['preexistencias'].copy(),
    {
        'id_usuario': 'usuario_origen',
        'familiar': 'familiar_original',
        'enfermedad': 'enfermedad_original',
    },
)
fact_antecedente_familiar = enriquecer_usuario(fact_antecedente_familiar)
fact_antecedente_familiar = derivar(
    fact_antecedente_familiar,
    'antecedente_id',
    lambda r: 'AF:'
    + hashlib.sha256(
        json.dumps(
            [r['usuario_origen'], r['familiar_original'], r['enfermedad_original']],
            ensure_ascii=False,
        ).encode('utf8')
    ).hexdigest(),
)
for c in ['familiar', 'enfermedad']:
    fact_antecedente_familiar = derivar(
        fact_antecedente_familiar, c, lambda r, c=c: limpiar(r[c + '_original'])
    )
fact_antecedente_familiar = derivar(
    fact_antecedente_familiar, 'es_antecedente_familiar', lambda r: True
)
fact_antecedente_familiar = S.select_columns(
    fact_antecedente_familiar,
    'fecha_nacimiento',
    'fecha_afiliacion_propia',
    complement=True,
    show=0,
)

# Eliminar únicamente relaciones IPS-droguería repetidas.
puente_ips_drogueria = renombrar(
    T.drop_duplicates(
        fuentes['ips_drogueria'], subset=['id_ips', 'id_drogueria'], show=0
    ),
    {'id_ips': 'ips_id', 'id_drogueria': 'drogueria_id'},
)
print('Hechos y puentes preparados. Ninguna tabla escrita todavía.')


Hechos y puentes preparados. Ninguna tabla escrita todavía.


## B6. Contratos del modelo y validación completa antes de escribir

Las claves y referencias de todas las tablas se validan en memoria, incluidas las opcionales no nulas.
Se contrastan eventos por fuente, edades, importes exactos, diagnósticos, tarifas y medicamentos.
El calendario cubre todos los roles usados en hechos. La fuente se vuelve a leer para detectar cambios.

**Funciones utilizadas en la siguiente celda**

- **Core, directamente o a través de auxiliares:** `DB_Extractor.execute_query`, `DataSelect.filter_in_list`, `DataSelect.filter_is_null`, `DataSelect.select_columns`, `TransformOperations.drop_duplicates`, `etl.schema.missing (alias nulo)`.
- **Fuera del Core y motivo:** Python configura PK/referencias lógicas, compara conteos y sumas Decimal y valida reglas del dominio. Los filtros de nulos, pertenencia y duplicados usan el Core; pandas solo presenta el resumen.

**Funciones propias definidas o llamadas en esta celda**

| Función | Relación con el Core y motivo de conservarla |
| --- | --- |
| `comprobar_fuente_sin_cambios` | relee mediante el Core y compara hashes de datos y archivos. |
| `exigir` | comprueba condiciones y emite errores de negocio del taller. |
| `huella_core` | calcula SHA-256 de archivos Python; no es una operación tabular del Core. |
| `huella_frame` | calcula SHA-256 de una serialización ordenada para comprobar reproducibilidad. |
| `minutos` | define diferencias y redondeo Decimal de seis posiciones para las esperas. |
| `q` | delega el SQL a DB_Extractor.execute_query y silencia sus mensajes. |
| `registrar_estado` | escribe el manifiesto específico de este proceso en JSON. |
| `sello` | combina fecha y hora locales; el calendario Core no proporciona este auxiliar escalar. |
| `unico` | delega selección, nulos y deduplicación al Core y añade los mensajes/valores inválidos del contrato. |

**Llamadas externas directas de esta celda** (incluye los cuerpos de funciones definidos aquí; las dependencias de auxiliares llamados se explican donde se definen; `…` omite argumentos):

| Grupo | Funciones y métodos | Por qué se conservan fuera del Core |
| --- | --- | --- |
| Python estándar y contenedores | `PK.update`, `REFS.extend`, `all`, `consultas_extraccion.items`, `conteos_esperados.items`, `esperas_origen.append`, `globals`, `int`, `len`, `list`, `modelo.items`, `print`, `set`, `str`, `str.join`, `sum`, `tuple`, `validacion_refs.append`, `zip` | Orquestación, condiciones, iteración, construcción de SQL, parámetros y mensajes. No sustituyen una transformación tabular del Core. |
| DataFrames y matrices | `dim_paciente.to_dict`, `fuentes[…].itertuples`, `fuentes[…].to_dict`, `pd.DataFrame` | Construcción de evidencia, acceso/iteración de resultados, selección de máximos y presentación de matrices. Los filtros, CSV, conversiones y deduplicación con equivalencia directa se delegan al Core. head del Core devuelve texto; no hay pivot/reindex tabular equivalente en estos transformadores. |
| Dibujo e IPython | `display` | Renderizado, ejes, leyendas, imágenes y visualización del notebook; el Core no ofrece renderizadores Python de gráficos ni display HTML. |
| Expresiones escalares del dominio | `Decimal` | Normalización Unicode, fechas, duraciones, claves y redondeo Decimal de las reglas explícitas. Se aplican con callables Core cuando corresponden a columnas; no son funciones propias del Core. |

In [17]:
# Contratos y conciliación completa antes de escribir.


# Reunir las tablas del modelo y declarar sus claves lógicas.
dimensiones = {
    k: globals()[k]
    for k in [
        'dim_fecha',
        'dim_empresa',
        'dim_ips',
        'dim_medico',
        'dim_familia',
        'dim_paciente',
        'dim_servicio',
        'dim_medicamento',
        'dim_diagnostico',
        'dim_drogueria',
    ]
}
hechos = {
    k: globals()[k]
    for k in [
        'puente_ips_drogueria',
        'fact_atencion',
        'fact_pago',
        'fact_formula',
        'puente_formula_medicamento',
        'fact_retiro',
        'fact_antecedente_familiar',
    ]
}
modelo = {**dimensiones, **hechos}
PK = {
    t: [c]
    for t, c in [
        ('dim_fecha', 'fecha_id'),
        ('dim_empresa', 'empresa_id'),
        ('dim_ips', 'ips_id'),
        ('dim_medico', 'medico_id'),
        ('dim_familia', 'familia_id'),
        ('dim_paciente', 'paciente_id'),
        ('dim_servicio', 'servicio_id'),
        ('dim_medicamento', 'medicamento_id'),
        ('dim_diagnostico', 'diagnostico_id'),
        ('dim_drogueria', 'drogueria_id'),
        ('fact_atencion', 'atencion_id'),
        ('fact_pago', 'id_transaccion'),
        ('fact_formula', 'codigo_formula'),
        ('fact_retiro', 'id_retiro'),
        ('fact_antecedente_familiar', 'antecedente_id'),
    ]
}
PK.update(
    {
        'puente_ips_drogueria': ['ips_id', 'drogueria_id'],
        'puente_formula_medicamento': ['codigo_formula', 'medicamento_id'],
    }
)
# Tupla: tabla, columna, dimensión/tabla destino, clave destino, permite nulo.

# Declarar cada referencia y si permite nulos; aún no son restricciones físicas.
REFS = [
    ('dim_medico', 'ips_id', 'dim_ips', 'ips_id', False),
    ('dim_familia', 'paciente_titular_id', 'dim_paciente', 'paciente_id', False),
    ('dim_paciente', 'familia_id', 'dim_familia', 'familia_id', False),
    ('dim_paciente', 'paciente_titular_id', 'dim_paciente', 'paciente_id', False),
    ('dim_paciente', 'empresa_id', 'dim_empresa', 'empresa_id', True),
    ('dim_paciente', 'ips_afiliacion_id', 'dim_ips', 'ips_id', False),
    ('puente_ips_drogueria', 'ips_id', 'dim_ips', 'ips_id', False),
    ('puente_ips_drogueria', 'drogueria_id', 'dim_drogueria', 'drogueria_id', False),
    ('fact_atencion', 'medico_id', 'dim_medico', 'medico_id', False),
    ('fact_atencion', 'medico_remitente_id', 'dim_medico', 'medico_id', True),
    ('fact_atencion', 'ips_medico_asociado_id', 'dim_ips', 'ips_id', False),
    ('fact_atencion', 'servicio_id', 'dim_servicio', 'servicio_id', True),
    ('fact_atencion', 'diagnostico_id', 'dim_diagnostico', 'diagnostico_id', False),
    ('fact_atencion', 'fecha_solicitud_id', 'dim_fecha', 'fecha_id', False),
    ('fact_atencion', 'fecha_atencion_id', 'dim_fecha', 'fecha_id', False),
    ('fact_pago', 'fecha_pago_id', 'dim_fecha', 'fecha_id', False),
    ('fact_formula', 'fecha_formula_id', 'dim_fecha', 'fecha_id', False),
    ('fact_formula', 'medico_id', 'dim_medico', 'medico_id', False),
    ('fact_formula', 'ips_medico_asociado_id', 'dim_ips', 'ips_id', False),
    ('fact_retiro', 'fecha_retiro_id', 'dim_fecha', 'fecha_id', False),
    (
        'puente_formula_medicamento',
        'codigo_formula',
        'fact_formula',
        'codigo_formula',
        False,
    ),
    (
        'puente_formula_medicamento',
        'medicamento_id',
        'dim_medicamento',
        'medicamento_id',
        False,
    ),
]
for t in [
    'fact_atencion',
    'fact_pago',
    'fact_formula',
    'fact_retiro',
    'fact_antecedente_familiar',
]:
    REFS.extend(
        (t, c, d, k, n)
        for c, d, k, n in [
            ('paciente_id', 'dim_paciente', 'paciente_id', False),
            ('familia_id', 'dim_familia', 'familia_id', False),
            ('empresa_id', 'dim_empresa', 'empresa_id', True),
            ('ips_afiliacion_id', 'dim_ips', 'ips_id', False),
        ]
    )

for t, df in modelo.items():
    unico(df, PK[t], t)

# Comprobar correspondencia de referencias con filtros del Core.
validacion_refs = []
for t, c, d, k, opcional in REFS:
    valores = list(modelo[t][c])
    destino = set(modelo[d][k])
    presentes = S.filter_is_null(modelo[t], c, complement=True, show=0)
    faltan = len(S.filter_in_list(presentes, c, destino, complement=True, show=0))
    nulos = len(S.filter_is_null(modelo[t], c, show=0))
    exigir(
        faltan == 0 and (opcional or nulos == 0), 'Referencia inválida: ' + t + '.' + c
    )
    exigir(
        all(nulo(v) or str(v) not in ('None', 'nan', '<NA>', '') for v in valores),
        'Nulo convertido en texto: ' + t + '.' + c,
    )
    validacion_refs.append(
        {
            'tabla': t,
            'columna': c,
            'destino': d,
            'clave_destino': k,
            'opcional': opcional,
            'huerfanos': faltan,
            'nulos': nulos,
        }
    )


# Derivar del origen los conteos que debe conservar cada tabla transformada.
conteos_esperados = {
    'dim_fecha': len(dim_fecha),
    'dim_empresa': CONTEOS_ORIGEN['empresa'],
    'dim_ips': CONTEOS_ORIGEN['ips'],
    'dim_medico': CONTEOS_ORIGEN['medico'],
    'dim_familia': CONTEOS_ORIGEN['cotizante'],
    'dim_paciente': CONTEOS_ORIGEN['cotizante'] + CONTEOS_ORIGEN['beneficiario'],
    'dim_servicio': CONTEOS_ORIGEN['servicios_pos'],
    'dim_medicamento': len(set(detalle_formula['medicamento_id'])),
    'dim_diagnostico': len(set(texto_diag['diagnostico_normalizado'])),
    'dim_drogueria': CONTEOS_ORIGEN['drogueria'],
    'puente_ips_drogueria': len(
        set(
            tuple(r)
            for r in fuentes['ips_drogueria'].itertuples(index=False, name=None)
        )
    ),
    'fact_atencion': sum(
        CONTEOS_ORIGEN[t]
        for t in ['citas_generales', 'urgencias', 'hospitalizaciones', 'remisiones']
    ),
    'fact_pago': CONTEOS_ORIGEN['pagos'],
    'fact_formula': CONTEOS_ORIGEN['formulas_medicas'],
    'puente_formula_medicamento': len(
        set(zip(detalle_formula['codigo_formula'], detalle_formula['medicamento_id']))
    ),
    'fact_retiro': CONTEOS_ORIGEN['retiros'],
    'fact_antecedente_familiar': CONTEOS_ORIGEN['preexistencias'],
}
for t, n in conteos_esperados.items():
    exigir(len(modelo[t]) == n, 'Conteo transformado incorrecto: ' + t)
for t in ['citas_generales', 'urgencias', 'hospitalizaciones', 'remisiones']:
    exigir(
        sum(v == t for v in fact_atencion['tabla_origen']) == CONTEOS_ORIGEN[t],
        'Pérdida de eventos: ' + t,
    )

# Conciliar importes exactos antes de cualquier escritura.
TOTAL_APORTES_ORIGEN = sum(
    (Decimal(int(v)) for v in fuentes['pagos']['valor_pagado']), Decimal(0)
)
exigir(
    sum(fact_pago['valor_aporte'], Decimal(0)) == TOTAL_APORTES_ORIGEN,
    'Diferencia monetaria antes de carga',
)

# Recalcular esperas del origen para contrastar exclusiones y banderas.
esperas_origen = []
for t in ['citas_generales', 'urgencias', 'hospitalizaciones', 'remisiones']:
    for r in fuentes[t].to_dict('records'):
        f, h = (
            ('fecha_remision', 'hora_remision')
            if t == 'remisiones'
            else ('fecha_solicitud', 'hora_solicitud')
        )
        esperas_origen.append(
            minutos(sello(r[f], r[h]), sello(r['fecha_atencion'], r['hora_atencion']))
        )
NEGATIVAS_ORIGEN = sum(v < 0 for v in esperas_origen if v is not None)
exigir(
    sum(fact_atencion['espera_negativa']) == NEGATIVAS_ORIGEN,
    'No concilian esperas negativas',
)
exigir(
    sum(nulo(v) for v in fact_atencion['espera_valida_min'])
    == sum(v is None or v < 0 for v in esperas_origen),
    'La espera válida no conserva las exclusiones',
)
exigir(
    all(nulo(v) or 0 <= v <= 120 for v in fact_atencion['edad_al_evento']),
    'Edad fuera de rango',
)
exigir(
    sum(fact_atencion['tarifa_informada']) == CONTEOS_ORIGEN['remisiones'],
    'Cobertura de tarifas inesperada',
)
# Ausencia de atributos propios en beneficiarios: desconocido no significa False.
for r in dim_paciente.to_dict('records'):
    if r['tipo_registro'] == 'beneficiario':
        exigir(
            all(
                nulo(r[c])
                for c in [
                    'estrato_propio',
                    'salario_base_propio',
                    'fecha_afiliacion_propia',
                    'escolaridad_propia',
                    'tipo_cotizante_propio',
                    'proviene_otra_eps_propio',
                ]
            ),
            'Atributos propios de beneficiario inventados por una conversión',
        )
exigir(
    all(nulo(v) for v in dim_medicamento['precio']),
    'Se inventaron precios de medicamentos',
)
exigir(
    all(nulo(v) for v in dim_medicamento['nombre']),
    'Se inventaron nombres de medicamentos',
)


def comprobar_fuente_sin_cambios():
    # Releer las fuentes y comparar sus huellas antes de publicar el resultado.
    for t, sql in consultas_extraccion.items():
        exigir(
            huella_frame(q(ex_origen, sql)) == HUELLAS_ORIGEN[t],
            'El origen cambió durante la ejecución: ' + t,
        )
    exigir(huella_core() == CORE_ANTES, 'El Core cambió durante la ejecución')


comprobar_fuente_sin_cambios()
estado_run['estado'] = 'preparado_validado'
registrar_estado()
display(
    pd.DataFrame(
        [
            {'tabla': t, 'filas': len(df), 'claves_logicas': ', '.join(PK[t])}
            for t, df in modelo.items()
        ]
    )
)
print(
    'Referencias validadas:',
    len(REFS),
    '| esperas negativas conservadas:',
    NEGATIVAS_ORIGEN,
)
print('Total de aportes conciliado:', str(TOTAL_APORTES_ORIGEN))


,tabla,filas,claves_logicas
0,dim_fecha,1096,fecha_id
1,dim_empresa,21,empresa_id
2,dim_ips,86,ips_id
3,dim_medico,720,medico_id
4,dim_familia,1100,familia_id
5,dim_paciente,4416,paciente_id
6,dim_servicio,67,servicio_id
7,dim_medicamento,166,medicamento_id
8,dim_diagnostico,87,diagnostico_id
9,dim_drogueria,35,drogueria_id


Referencias validadas: 42 | esperas negativas conservadas: 497
Total de aportes conciliado: 8548421149


## B7. Escritura Core y conciliación del destino

Solo esta celda escribe, usando `DB_Loader.load_dimension` y `load_fact`, siempre con `if_exists='fail'`.
Las dimensiones quedan sin restricciones físicas. El mapa del loader solo contiene FK obligatorias no nulas.
Los importes de hechos se materializan como NUMERIC(18,2) y las esperas como NUMERIC(18,6).
Las fechas de las dimensiones, inferidas por el loader, son timestamp sin zona; las de hechos usan DATE/TIME.

El manifiesto exitoso solo se publica al conciliar todas las tablas, claves, referencias, nulos, tipos,
eventos, importes y huellas del origen. Ante un fallo se registra carga incompleta y no se reintenta ni limpia
el destino automáticamente. El origen siempre permanece READ ONLY.

**Funciones utilizadas en la siguiente celda**

- **Core, directamente o a través de auxiliares:** `CSV_Loader.save_csv`, `DB_Extractor.close_connection`, `DB_Extractor.execute_query`, `DB_Loader.load_dimension`, `DB_Loader.load_fact`, `DataSelect.filter_is_null`, `DataSelect.select_columns`, `etl.DB_Loader`.
- **Fuera del Core y motivo:** sqlalchemy.types define dtype, incluidos TIME y precisiones; no ejecuta la carga. SQL de conciliación calcula conteos/nulos/totales en destino mediante el extractor. JSON, Path, datetime y version publican evidencias. Se conserva DB_Loader con claves lógicas: persist_schema existe, pero adoptarlo cambiaría el contrato publicado y no se hace en esta refactorización.

**Funciones propias definidas o llamadas en esta celda**

| Función | Relación con el Core y motivo de conservarla |
| --- | --- |
| `comprobar_fuente_sin_cambios` | relee mediante el Core y compara hashes de datos y archivos. |
| `exigir` | comprueba condiciones y emite errores de negocio del taller. |
| `guardar_etl` | delega la exportación a CSV_Loader.save_csv. |
| `ident` | escapa identificadores SQL del catálogo; no calcula medidas. |
| `proyectar` | es un envoltorio de DataSelect.select_columns. |
| `q` | delega el SQL a DB_Extractor.execute_query y silencia sus mensajes. |
| `registrar_estado` | escribe el manifiesto específico de este proceso en JSON. |
| `tabla` | construye el nombre SQL public.tabla con identificadores escapados. |
| `tipos_hecho` | define los tipos SQL del modelo, incluido TIME; los pasa al loader Core sin escribir mediante SQLAlchemy. |

**SQL:** las consultas de esta celda se ejecutan mediante `DB_Extractor.execute_query`, pero sus expresiones SQL pertenecen al cuaderno. Se conserva SQL para perfilado, validación o conciliación; la extracción B1 solo selecciona columnas y la transformación tabular de B2–B5 usa los transformadores Core.

**Llamadas externas directas de esta celda** (incluye los cuerpos de funciones definidos aquí; las dependencias de auxiliares llamados se explican donde se definen; `…` omite argumentos):

| Grupo | Funciones y métodos | Por qué se conservan fuera del Core |
| --- | --- | --- |
| Python estándar y contenedores | `RuntimeError`, `claves_db.append`, `conteos_db.append`, `dimensiones.items`, `enumerate`, `estado_run.update`, `estado_run[…].append`, `hechos.items`, `int`, `len`, `list`, `modelo.items`, `next`, `nulos_db.append`, `print`, `refs_db.append`, `set`, `str`, `str.join`, `temporal.replace`, `type` | Orquestación, condiciones, iteración, construcción de SQL, parámetros y mensajes. No sustituyen una transformación tabular del Core. |
| Archivos, configuración y reproducibilidad | `ZoneInfo`, `datetime.now`, `datetime.now(…).isoformat`, `json.dumps`, `json.loads`, `publicado.exists`, `publicado.read_text`, `revision.mkdir`, `ruta.write_text`, `temporal.write_text`, `version` | Rutas, entorno, manifiestos, fechas de ejecución, versiones o hashes específicos de este proceso; el Core no publica estos manifiestos del taller. |
| DataFrames y matrices | `df.copy`, `eventos_db.to_dict`, `pd.DataFrame`, `tipos_db.to_dict` | Construcción de evidencia, acceso/iteración de resultados, selección de máximos y presentación de matrices. Los filtros, CSV, conversiones y deduplicación con equivalencia directa se delegan al Core. head del Core devuelve texto; no hay pivot/reindex tabular equivalente en estos transformadores. |
| Dibujo e IPython | `display` | Renderizado, ejes, leyendas, imágenes y visualización del notebook; el Core no ofrece renderizadores Python de gráficos ni display HTML. |
| Expresiones escalares del dominio | `Decimal`, `sqltypes.BigInteger`, `sqltypes.Boolean`, `sqltypes.Date`, `sqltypes.DateTime`, `sqltypes.Numeric`, `sqltypes.Text`, `sqltypes.Time` | Normalización Unicode, fechas, duraciones, claves y redondeo Decimal de las reglas explícitas. Se aplican con callables Core cuando corresponden a columnas; no son funciones propias del Core. |

In [18]:
# Carga inicial, conciliación persistida y publicación del manifiesto.


def guardar_etl(nombre, frame):
    # Exportar solo la evidencia seleccionada mediante el loader CSV del Core.
    CSV_CORE.save_csv(
        frame,
        str(SALIDA_ETL / (nombre + '.csv')),
        index=False,
        encoding='utf-8-sig',
        lineterminator='\n',
    )


def tipos_hecho(t, frame):
    # Asignar tipos por significado: fechas, horas, importes, esperas y banderas.
    fechas = {
        'fecha_solicitud',
        'fecha_atencion',
        'fecha_pago',
        'fecha_formula',
        'fecha_retiro',
        'fecha_afiliacion_propia',
    }
    horas = {'hora_solicitud', 'hora_atencion'}
    stamps = {'timestamp_solicitud_local', 'timestamp_atencion_local'}
    dinero_cols = {'valor_aporte', 'tarifa_servicio_catalogo'}
    esperas = {'espera_minutos_cruda', 'espera_valida_min'}
    enteros = {
        'duracion_hospitalizacion',
        'edad_solicitud',
        'edad_al_evento',
        'edad_al_retiro',
        'dias_afiliacion_al_retiro',
        'presencia_en_receta',
    }
    booleanos = {
        'espera_incompleta',
        'espera_negativa',
        'tarifa_informada',
        'duracion_unidad_informada',
        'ips_medico_sin_vigencia_historica',
        'destino_informado',
        'es_antecedente_familiar',
    }
    # Respetar esta prioridad si una columna aparece en más de un grupo.
    return {
        c: (
            sqltypes.Date() if c in fechas
            else sqltypes.Time() if c in horas
            else sqltypes.DateTime(timezone=False) if c in stamps
            else sqltypes.Numeric(18, 2) if c in dinero_cols
            else sqltypes.Numeric(18, 6) if c in esperas
            else sqltypes.BigInteger() if c in enteros
            else sqltypes.Boolean() if c in booleanos
            else sqltypes.Text()
        )
        for c in frame.columns
    }


try:
    if not EJECUTAR_CARGA:
        # En modo preparación, guardar evidencia separada de la carga publicada.
        revision = SALIDA_ETL / 'preparaciones' / RUN_ID
        revision.mkdir(parents=True, exist_ok=True)
        for nombre, frame in [
            (
                'mapeo_servicios',
                proyectar(
                    dim_servicio,
                    'servicio_id',
                    'descripcion_original',
                    'descripcion',
                    'tipo_servicio',
                    'tarifa_catalogo',
                ),
            ),
            ('mapeo_diagnosticos', mapeo_diagnosticos),
            ('clasificacion_diagnosticos', dim_diagnostico),
        ]:
            CSV_CORE.save_csv(
                frame,
                str(revision / (nombre + '.csv')),
                index=False,
                encoding='utf-8-sig',
                lineterminator='\n',
            )
        estado_run['estado'] = 'preparacion_exitosa_sin_escrituras'
        registrar_estado()
        (SALIDA_ETL / 'preparacion_etl.json').write_text(
            json.dumps(
                {
                    **estado_run,
                    'conteos': conteos_esperados,
                    'referencias_validadas': len(REFS),
                    'core_sha256': CORE_ANTES,
                },
                ensure_ascii=False,
                indent=2,
            ),
            encoding='utf8',
        )
        publicado = SALIDA_ETL / 'manifest_carga.json'
        if publicado.exists():
            anterior = json.loads(publicado.read_text(encoding='utf8'))
            print(
                'Preparación sin escrituras. Carga publicada:',
                anterior['run_id'],
                '| estado:',
                anterior['estado'],
            )
            print(
                'La verificación independiente de la bodega está en'
                ' carga_core/verificacion_independiente.json.'
            )
        else:
            print(
                'Preparación terminada. No se escribieron tablas. Use ejecutar_etl.py'
                ' --cargar para la carga inicial.'
            )
    else:
        # Se repite la guarda inmediatamente antes de crear la primera tabla.
        exigir(
            len(q(ex_destino, catalogo_sql)) == 0,
            'El catálogo destino cambió antes de cargar.',
        )
        estado_run['estado'] = 'cargando'
        registrar_estado()

        # Escribir solo tras comprobar que el catálogo de destino sigue vacío.
        loader = DB_Loader(ex_destino.engine)
        for t, df in dimensiones.items():
            loader.load_dimension(df.copy(), t, if_exists='fail', index=False)
            estado_run['tablas_cargadas'].append(t)
            registrar_estado()
        for t, df in hechos.items():

            # Delegar al loader la validación de referencias obligatorias no nulas.
            mapa = {
                c: (d, k) for rt, c, d, k, opcional in REFS if rt == t and not opcional
            }
            loader.load_fact(
                df.copy(),
                t,
                if_exists='fail',
                index=False,
                foreign_keys_map=mapa,
                validate_foreign_keys=True,
                dtype=tipos_hecho(t, df),
            )
            estado_run['tablas_cargadas'].append(t)
            registrar_estado()

        # Conciliación persistida, solo SELECT y sin descargar ni mostrar personas.
        estado_run['estado'] = 'conciliando'
        registrar_estado()

        # Iniciar las comprobaciones del contenido realmente persistido.
        conteos_db = []
        claves_db = []
        nulos_db = []
        for t, df in modelo.items():
            n = int(q(ex_destino, f'SELECT COUNT(*) AS n FROM {tabla(t)}').iloc[0, 0])
            exigir(n == len(df), 'No concilia cantidad en ' + t)
            conteos_db.append(
                {'tabla': t, 'esperado': len(df), 'cargado': n, 'ok': True}
            )
            columnas_pk = ', '.join(ident(c) for c in PK[t])
            nulos_pk = ' OR '.join(ident(c) + ' IS NULL' for c in PK[t])
            duplicadas = int(
                q(
                    ex_destino,
                    f'SELECT COUNT(*) FROM (SELECT {columnas_pk} FROM {tabla(t)} GROUP'
                    f' BY {columnas_pk} HAVING COUNT(*)>1) x',
                ).iloc[0, 0]
            )
            sin_pk = int(
                q(ex_destino, f'SELECT COUNT(*) FROM {tabla(t)} WHERE {nulos_pk}').iloc[
                    0, 0
                ]
            )
            exigir(duplicadas == 0 and sin_pk == 0, 'Clave inválida en destino ' + t)
            claves_db.append(
                {
                    'tabla': t,
                    'clave_logica': ','.join(PK[t]),
                    'duplicadas': duplicadas,
                    'nulos': sin_pk,
                }
            )
            expr = ', '.join(
                f'COUNT(*) FILTER(WHERE {ident(c)} IS NULL) AS n{i}'
                for i, c in enumerate(df.columns)
            )
            counts = q(ex_destino, f'SELECT {expr} FROM {tabla(t)}').iloc[0]
            for i, c in enumerate(df.columns):
                esperado = len(S.filter_is_null(df, c, show=0))
                real = int(counts[f'n{i}'])
                exigir(esperado == real, 'Diferencia de nulos: ' + t + '.' + c)
                nulos_db.append(
                    {
                        'tabla': t,
                        'columna': c,
                        'nulos_esperados': esperado,
                        'nulos_cargados': real,
                    }
                )

        # Verificar en destino que ninguna referencia no nula quede huérfana.
        refs_db = []
        for t, c, d, k, opcional in REFS:
            sql = (
                f'SELECT COUNT(*) FROM {tabla(t)} h WHERE h.{ident(c)} IS NOT NULL AND'
                ' NOT EXISTS '
            )
            sql += f'(SELECT 1 FROM {tabla(d)} d WHERE h.{ident(c)}=d.{ident(k)})'
            h = int(q(ex_destino, sql).iloc[0, 0])
            exigir(h == 0, 'Huérfanos después de carga: ' + t + '.' + c)
            refs_db.append(
                {
                    'tabla': t,
                    'columna': c,
                    'destino': d,
                    'clave_destino': k,
                    'opcional': opcional,
                    'huerfanos': h,
                }
            )

        # Comprobar tipos y escalas de importes/esperas tras la materialización.
        tipos_db = q(
            ex_destino,
            """SELECT table_name AS tabla,column_name AS columna,data_type AS tipo,
              numeric_precision AS precision,numeric_scale AS escala FROM information_schema.columns
              WHERE table_schema='public' ORDER BY table_name,ordinal_position""",
        )
        for t, c, escala in [
            ('fact_pago', 'valor_aporte', 2),
            ('fact_atencion', 'tarifa_servicio_catalogo', 2),
            ('fact_atencion', 'espera_minutos_cruda', 6),
            ('fact_atencion', 'espera_valida_min', 6),
        ]:
            fila = next(
                r
                for r in tipos_db.to_dict('records')
                if r['tabla'] == t and r['columna'] == c
            )
            exigir(
                fila['tipo'] == 'numeric'
                and int(fila['precision']) == 18
                and int(fila['escala']) == escala,
                'Tipo monetario/temporal incorrecto',
            )

        # Conciliar aportes exactos entre origen, transformación y destino.
        total_db = Decimal(
            q(ex_destino, 'SELECT SUM(valor_aporte)::text FROM public.fact_pago').iloc[
                0, 0
            ]
        )
        total_origen_sql = Decimal(
            q(ex_origen, 'SELECT SUM(valor_pagado)::text FROM public.pagos').iloc[0, 0]
        )
        exigir(
            total_db == TOTAL_APORTES_ORIGEN == total_origen_sql,
            'Diferencia en total exacto de aportes',
        )
        eventos_db = q(
            ex_destino,
            'SELECT tabla_origen,COUNT(*) AS filas FROM public.fact_atencion GROUP BY'
            ' tabla_origen ORDER BY tabla_origen',
        )
        for r in eventos_db.to_dict('records'):
            exigir(
                int(r['filas']) == CONTEOS_ORIGEN[r['tabla_origen']],
                'Diferencia por tipo de atención',
            )

        # Comprobar que se conservaron exclusiones temporales y cobertura de tarifas.
        calidad_db = q(
            ex_destino,
            """SELECT COUNT(*) AS eventos,
            COUNT(*) FILTER(WHERE espera_negativa) AS negativas,
            COUNT(*) FILTER(WHERE espera_incompleta) AS incompletas,
            COUNT(*) FILTER(WHERE espera_valida_min IS NOT NULL) AS validas,
            COUNT(*) FILTER(WHERE espera_negativa AND espera_valida_min IS NOT NULL) AS negativas_incluidas,
            COUNT(*) FILTER(WHERE tarifa_informada) AS eventos_con_tarifa,
            COUNT(*) FILTER(WHERE tabla_origen<>'remisiones' AND tarifa_servicio_catalogo IS NOT NULL) AS tarifa_inventada
            FROM public.fact_atencion""",
        )
        calidad = calidad_db.iloc[0]
        exigir(
            int(calidad['negativas']) == NEGATIVAS_ORIGEN
            and int(calidad['negativas_incluidas']) == 0
            and int(calidad['tarifa_inventada']) == 0,
            'Calidad de espera/tarifas no conciliada',
        )

        # Contrastar las tarifas de remisiones contra el catálogo de origen.
        tarifa_origen = Decimal(
            q(
                ex_origen,
                'SELECT SUM(s.costo)::text FROM public.remisiones r JOIN'
                ' public.servicios_pos s ON s.id_servicio_pos=r.servicio_pos',
            ).iloc[0, 0]
        )
        tarifa_db = Decimal(
            q(
                ex_destino,
                'SELECT SUM(tarifa_servicio_catalogo)::text FROM public.fact_atencion',
            ).iloc[0, 0]
        )
        exigir(tarifa_origen == tarifa_db, 'Diferencia en tarifas de remisiones')
        # Conciliación por evento y valores: no basta comparar únicamente sumas
        # globales.
        tarifas_distintas = q(
            ex_destino,
            """SELECT COUNT(*) FROM public.fact_atencion a
            JOIN public.dim_servicio s ON s.servicio_id=a.servicio_id
            WHERE a.tarifa_servicio_catalogo IS DISTINCT FROM s.tarifa_catalogo""",
        ).iloc[0, 0]
        exigir(int(tarifas_distintas) == 0, 'Tarifa de evento no coincide con catálogo')
        comprobar_fuente_sin_cambios()
        exigir(
            set(q(ex_destino, catalogo_sql)['tabla']) == set(modelo),
            'Catálogo final inesperado',
        )
        # Los loaders no imponen restricciones físicas: conservar constancia
        # verificable.

        # Documentar que esta ruta conserva el contrato de relaciones lógicas.
        restricciones = q(
            ex_destino,
            """SELECT table_name,constraint_type FROM information_schema.table_constraints
                  WHERE table_schema='public' AND constraint_type IN ('PRIMARY KEY','FOREIGN KEY','UNIQUE')""",
        )
        exigir(
            len(restricciones) == 0,
            'Se observaron restricciones no producidas por esta carga Core',
        )

        guardar_etl(
            'mapeo_servicios',
            proyectar(
                dim_servicio,
                'servicio_id',
                'descripcion_original',
                'descripcion',
                'tipo_servicio',
                'tarifa_catalogo',
            ),
        )
        guardar_etl('mapeo_diagnosticos', mapeo_diagnosticos)
        guardar_etl('clasificacion_diagnosticos', dim_diagnostico)
        guardar_etl('conteos_carga', pd.DataFrame(conteos_db))
        guardar_etl('claves_logicas', pd.DataFrame(claves_db))
        guardar_etl('referencias_carga', pd.DataFrame(refs_db))
        guardar_etl('nulos_carga', pd.DataFrame(nulos_db))
        guardar_etl('diccionario_bodega', tipos_db)
        guardar_etl('eventos_por_fuente', eventos_db)
        guardar_etl('calidad_temporal_tarifas', calidad_db)
        guardar_etl(
            'conciliacion_monetaria',
            pd.DataFrame(
                [
                    {
                        'medida': 'aportes',
                        'origen': str(total_origen_sql),
                        'destino': str(total_db),
                        'ok': True,
                    },
                    {
                        'medida': 'tarifas_remisiones',
                        'origen': str(tarifa_origen),
                        'destino': str(tarifa_db),
                        'ok': True,
                    },
                ]
            ),
        )

        # Publicar el contrato del modelo solo cuando todas las comprobaciones pasan.
        contrato = {
            'claves_logicas': PK,
            'referencias': [
                {
                    'tabla': t,
                    'columna': c,
                    'destino': d,
                    'clave_destino': k,
                    'opcional': n,
                }
                for t, c, d, k, n in REFS
            ],
            'restricciones_fisicas': False,
            'tablas': {t: list(df.columns) for t, df in modelo.items()},
        }
        (SALIDA_ETL / 'modelo_bodega.json').write_text(
            json.dumps(contrato, ensure_ascii=False, indent=2), encoding='utf8'
        )
        estado_run.update(
            {
                'estado': 'exitoso',
                'fin_bogota': datetime.now(ZoneInfo('America/Bogota')).isoformat(),
                'conciliacion_aprobada': True,
                'conteos': conteos_esperados,
                'huellas_origen_sha256': HUELLAS_ORIGEN,
                'origen_sin_cambios': True,
                'core_sha256': CORE_ANTES,
                'core_sin_cambios': True,
                'referencias_validadas': len(REFS),
                'restricciones_fisicas': False,
                'total_aportes': str(total_db),
                'total_tarifas_remisiones': str(tarifa_db),
                'esperas_negativas': NEGATIVAS_ORIGEN,
                'snapshot_global': False,
                'zona_timestamps_locales': 'America/Bogota',
                'moneda_confirmada': False,
                'paquetes': {
                    p: version(p)
                    for p in ['pandas', 'SQLAlchemy', 'psycopg2-binary', 'holidays']
                },
                'limites': [
                    'sin_costos_reales',
                    'sin_catalogo_medicamentos',
                    'sin_historia_perfiles',
                    'sin_capacidad_ocupacion',
                    'sin_datos_mora',
                    'sin_ddl_restricciones',
                    'sin_atomicidad_global',
                ],
            }
        )
        registrar_estado()
        # Publicar éxito al final, con reemplazo atómico del archivo local.

        # Sustituir el manifiesto exitoso mediante un reemplazo de archivo al final.
        temporal = SALIDA_ETL / 'manifest_carga.tmp'
        temporal.write_text(
            json.dumps(estado_run, ensure_ascii=False, indent=2), encoding='utf8'
        )
        temporal.replace(SALIDA_ETL / 'manifest_carga.json')
        display(pd.DataFrame(conteos_db))
        display(calidad_db)
        print(
            'Carga completa y conciliación aprobada. Manifiesto:'
            ' carga_core/manifest_carga.json'
        )
except Exception as error:

    # Registrar el fallo sin reintentar ni limpiar automáticamente el destino.
    estado_run.update(
        {
            'estado': (
                'carga_incompleta'
                if estado_run['tablas_cargadas']
                else 'fallo_sin_carga'
            ),
            'tipo_error': type(error).__name__,
        }
    )
    registrar_estado()
    # No propagar excepciones de drivers que puedan contener filas o parámetros.
    raise RuntimeError(
        'ETL no completado. No reintentar append/replace. Consulte el registro de'
        ' intento '
        + RUN_ID
    ) from None
finally:

    # Cerrar conexiones incluso si una comprobación o exportación falla.
    for ex in (ex_origen, ex_destino):
        if ex:
            ex.close_connection()


✅ Dimensión 'dim_fecha' cargada exitosamente en la base de datos 'postgresql (modo: fail).
✅ Dimensión 'dim_empresa' cargada exitosamente en la base de datos 'postgresql (modo: fail).
✅ Dimensión 'dim_ips' cargada exitosamente en la base de datos 'postgresql (modo: fail).
✅ Dimensión 'dim_medico' cargada exitosamente en la base de datos 'postgresql (modo: fail).
✅ Dimensión 'dim_familia' cargada exitosamente en la base de datos 'postgresql (modo: fail).


✅ Dimensión 'dim_paciente' cargada exitosamente en la base de datos 'postgresql (modo: fail).
✅ Dimensión 'dim_servicio' cargada exitosamente en la base de datos 'postgresql (modo: fail).
✅ Dimensión 'dim_medicamento' cargada exitosamente en la base de datos 'postgresql (modo: fail).
✅ Dimensión 'dim_diagnostico' cargada exitosamente en la base de datos 'postgresql (modo: fail).
✅ Dimensión 'dim_drogueria' cargada exitosamente en la base de datos 'postgresql (modo: fail).
✅ Hechos cargados exitosamente en la tabla 'puente_ips_drogueria' (modo: fail).


✅ Hechos cargados exitosamente en la tabla 'fact_atencion' (modo: fail).


✅ Hechos cargados exitosamente en la tabla 'fact_pago' (modo: fail).


✅ Hechos cargados exitosamente en la tabla 'fact_formula' (modo: fail).


...

✅ Hechos cargados exitosamente en la tabla 'fact_antecedente_familiar' (modo: fail).


💾 CSV guardado exitosamente: C:\Users\cabre\Desktop\University\Tesis\python-library-etl-master\python-library-etl-master\colombia_saludable\planes_y_evidencia\carga_core\mapeo_servicios.csv (67 registros, 5 columnas)
💾 CSV guardado exitosamente: C:\Users\cabre\Desktop\University\Tesis\python-library-etl-master\python-library-etl-master\colombia_saludable\planes_y_evidencia\carga_core\mapeo_diagnosticos.csv (90 registros, 3 columnas)
💾 CSV guardado exitosamente: C:\Users\cabre\Desktop\University\Tesis\python-library-etl-master\python-library-etl-master\colombia_saludable\planes_y_evidencia\carga_core\clasificacion_diagnosticos.csv (87 registros, 5 columnas)
💾 CSV guardado exitosamente: C:\Users\cabre\Desktop\University\Tesis\python-library-etl-master\python-library-etl-master\colombia_saludable\planes_y_evidencia\carga_core\conteos_carga.csv (17 registros, 4 columnas)
💾 CSV guardado exitosamente: C:\Users\cabre\Desktop\University\Tesis\python-library-etl-master\python-library-etl-master

,tabla,esperado,cargado,ok
0,dim_fecha,1096,1096,True
1,dim_empresa,21,21,True
2,dim_ips,86,86,True
3,dim_medico,720,720,True
4,dim_familia,1100,1100,True
5,dim_paciente,4416,4416,True
6,dim_servicio,67,67,True
7,dim_medicamento,166,166,True
8,dim_diagnostico,87,87,True
9,dim_drogueria,35,35,True


,eventos,negativas,incompletas,validas,negativas_incluidas,eventos_con_tarifa,tarifa_inventada
0,21207,497,0,20710,0,16742,0


Carga completa y conciliación aprobada. Manifiesto: carga_core/manifest_carga.json
🔒 Conexión cerrada.
🔒 Conexión cerrada.


## B8. Estado para el cuaderno de respuestas

Si la salida anterior indica carga completa, `planes_y_evidencia/carga_core/manifest_carga.json` debe contener
`estado="exitoso"` y `conciliacion_aprobada=true`. Ese es el contrato para desarrollar `answers.ipynb`.
La bodega contiene grano detallado y relaciones lógicas verificadas, con las limitaciones de datos del plan.
No se han implementado las respuestas en esta tarea.

Una repetición con `--cargar` se bloquea al inicio si existen objetos en destino. Una ejecución normal sin
esa opción prepara de nuevo en memoria y no cambia la bodega ni sustituye el manifiesto exitoso.
El diagnóstico puede ejecutarse por separado con `ejecutar_diagnostico.py`, que excluye las celdas ETL.